In [1]:

# ================================================================
# B2 — CELL 1: ENVIRONMENT, DATASET & EXPERIMENT PROTOCOL
# ================================================================

import os
import sys
import json
import platform
from pathlib import Path
from collections import Counter

import torch

print("=" * 76)
print("B2 — CELL 1: ENVIRONMENT AND EXPERIMENT VERIFICATION")
print("=" * 76)

# ------------------------------------------------
# 1. Project directories
# ------------------------------------------------

ROOT = Path("/kaggle/working/B2_FastV")

SUBDIRS = [
    "configs",
    "scripts",
    "manifests",
    "predictions",
    "metrics",
    "profiling",
    "logs",
    "checkpoints",
    "final_results",
]

ROOT.mkdir(parents=True, exist_ok=True)

for name in SUBDIRS:
    (ROOT / name).mkdir(parents=True, exist_ok=True)

print("\n[1] PROJECT DIRECTORIES")
print("Project root:", ROOT)
print("Directories created:", len(SUBDIRS))

# ------------------------------------------------
# 2. Runtime
# ------------------------------------------------

print("\n[2] PYTHON AND PYTORCH")

print("Python:", platform.python_version())
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

# ------------------------------------------------
# 3. GPU verification
# ------------------------------------------------

print("\n[3] GPU HARDWARE")

assert torch.cuda.is_available(), (
    "CUDA GPU unavailable. Enable GPU in "
    "Kaggle notebook settings."
)

GPU_COUNT = torch.cuda.device_count()

print("GPU count:", GPU_COUNT)

for i in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(i)

    print(f"\nGPU {i}")
    print("Name:", properties.name)
    print(
        "Total memory:",
        round(properties.total_memory / 1024**3, 3),
        "GiB"
    )
    print(
        "Compute capability:",
        f"{properties.major}.{properties.minor}"
    )

assert GPU_COUNT >= 2, (
    "This experiment's verified FP16 loader "
    "requires two GPUs. Select 2x T4 in Kaggle."
)

print("\nTwo-GPU requirement: PASS")

# ------------------------------------------------
# 4. Discover the attached SLAKE dataset
# ------------------------------------------------

print("\n[4] DISCOVER SLAKE DATASET")

INPUT_ROOT = Path("/kaggle/input")

assert INPUT_ROOT.is_dir(), (
    "Kaggle input directory not found."
)

print("\nAttached Kaggle inputs:")

for path in sorted(INPUT_ROOT.iterdir()):
    print(" ", path)

# Search for official SLAKE test.json, without
# assuming the uploader's dataset slug.

candidates = sorted(INPUT_ROOT.rglob("test.json"))

valid_candidates = []

for candidate in candidates:
    try:
        with candidate.open(
            "r", encoding="utf-8"
        ) as f:
            records = json.load(f)

        if not isinstance(records, list):
            continue

        if not records:
            continue

        first = records[0]

        required_fields = {
            "qid",
            "question",
            "answer",
            "answer_type",
            "img_name",
            "q_lang",
        }

        if not isinstance(first, dict):
            continue

        if not required_fields.issubset(first):
            continue

        english_records = [
            r for r in records
            if isinstance(r, dict)
            and str(
                r.get("q_lang", "")
            ).lower().strip() == "en"
        ]

        valid_candidates.append({
            "path": candidate,
            "records": records,
            "english": english_records,
        })

    except (OSError, ValueError, TypeError):
        continue

print(
    "\nSLAKE-format test files found:",
    len(valid_candidates)
)

for entry in valid_candidates:
    print(
        str(entry["path"]),
        "| total:", len(entry["records"]),
        "| English:", len(entry["english"])
    )

assert len(valid_candidates) == 1, (
    "Expected exactly one SLAKE test.json. "
    "If multiple or zero candidates are found, "
    "inspect the listed paths before continuing."
)

selected = valid_candidates[0]

TEST_JSON = selected["path"]
SLAKE_ROOT = TEST_JSON.parent
IMAGES_ROOT = SLAKE_ROOT / "imgs"

test_records = selected["records"]
english_records = selected["english"]

print("\nSelected dataset root:", SLAKE_ROOT)
print("Official test JSON:", TEST_JSON)
print("Image directory:", IMAGES_ROOT)

# ------------------------------------------------
# 5. Verify the official English test protocol
# ------------------------------------------------

print("\n[5] OFFICIAL SLAKE SPLIT VERIFICATION")

assert len(test_records) == 2094, (
    f"Expected 2094 total test questions, "
    f"found {len(test_records)}"
)

assert len(english_records) == 1061, (
    f"Expected 1061 English questions, "
    f"found {len(english_records)}"
)

type_counts = Counter(
    str(r["answer_type"]).lower().strip()
    for r in english_records
)

print("All-language test questions:", len(test_records))
print("English test questions:", len(english_records))
print("Answer types:", dict(type_counts))

assert type_counts["closed"] == 416
assert type_counts["open"] == 645
assert set(type_counts) == {"closed", "open"}

qids = [str(r["qid"]) for r in english_records]

assert len(qids) == len(set(qids)), (
    "Duplicate English question IDs detected."
)

assert IMAGES_ROOT.is_dir(), (
    "SLAKE image directory is missing."
)

missing_images = sorted({
    str(r["img_name"])
    for r in english_records
    if not (
        IMAGES_ROOT / str(r["img_name"])
    ).is_file()
})

assert not missing_images, (
    "Missing SLAKE images: "
    + repr(missing_images[:10])
)

print("Unique English QIDs:", len(set(qids)))
print("Missing images:", len(missing_images))
print("Official English test protocol: PASS")

# ------------------------------------------------
# 6. FastV retention configuration
# ------------------------------------------------

print("\n[6] FASTV CONFIGURATION")

ORIGINAL_VISUAL_TOKENS = 576
PRUNING_AFTER_LAYER = 2
NUM_DECODER_LAYERS = 32

RETENTIONS = (75, 50, 25)

RETAINED_COUNTS = {
    pct: ORIGINAL_VISUAL_TOKENS * pct // 100
    for pct in RETENTIONS
}

assert RETAINED_COUNTS == {
    75: 432,
    50: 288,
    25: 144
}

assert PRUNING_AFTER_LAYER == 2

assert NUM_DECODER_LAYERS == 32

for pct in RETENTIONS:
    print(
        f"{pct}% retention:"
        f" {RETAINED_COUNTS[pct]} visual tokens"
        f" | removed: "
        f"{576 - RETAINED_COUNTS[pct]}"
    )

EXPECTED_TOTAL = len(english_records) * len(RETENTIONS)

print("\nExpected B2 predictions:", EXPECTED_TOTAL)

assert EXPECTED_TOTAL == 3183

# ------------------------------------------------
# 7. Define exact B1-compatible metric protocol
# ------------------------------------------------

print("\n[7] B1-COMPATIBLE EVALUATION PROTOCOL")

METRICS_PROTOCOL = {
    "source": "B1 Cell 11B",
    "source_file": (
        "baselines/b1_results/"
        "b1-benchmark-final.ipynb"
    ),
    "normalization": (
        "Lowercase, trim, replace hyphens with "
        "spaces, remove punctuation, collapse whitespace"
    ),
    "closed_yes_no": (
        "Extract leading yes/no from normalized "
        "prediction and compare with reference"
    ),
    "closed_categorical": (
        "Normalized exact match only"
    ),
    "open": (
        "Counter-based token overlap: "
        "precision, recall, F1"
    ),
    "manual_semantic_grading": False,
    "llm_as_judge": False,
}

for key, value in METRICS_PROTOCOL.items():
    print(f"{key}: {value}")

# ------------------------------------------------
# 8. Save experiment configuration
# ------------------------------------------------

print("\n[8] SAVE EXPERIMENT CONFIGURATION")

EXPERIMENT = {
    "experiment": "B2_FastV",
    "model_id": "microsoft/llava-med-v1.5-mistral-7b",
    "model_revision": (
        "91bb16c122001ddc9cf1fd36ce1dae09448943a2"
    ),
    "method": "FastV",
    "pruning_after_layer": PRUNING_AFTER_LAYER,
    "num_decoder_layers": NUM_DECODER_LAYERS,
    "original_visual_tokens": ORIGINAL_VISUAL_TOKENS,
    "retention_percentages": list(RETENTIONS),
    "retained_visual_tokens": {
        str(k): v
        for k, v in RETAINED_COUNTS.items()
    },
    "dataset": "SLAKE 1.0",
    "split": "official_test",
    "language": "en",
    "test_json": str(TEST_JSON),
    "images_root": str(IMAGES_ROOT),
    "total_test_questions": 2094,
    "english_test_questions": 1061,
    "closed_questions": 416,
    "open_questions": 645,
    "expected_predictions": EXPECTED_TOTAL,
    "decoding": {
        "strategy": "greedy",
        "max_new_tokens": 64,
        "temperature": None,
        "do_sample": False,
    },
    "metrics": METRICS_PROTOCOL,
    "hardware": [
        {
            "index": i,
            "name": torch.cuda.get_device_name(i),
            "total_memory_gib": round(
                torch.cuda.get_device_properties(i)
                .total_memory / 1024**3, 3
            ),
        }
        for i in range(GPU_COUNT)
    ],
}

CONFIG_PATH = ROOT / "configs/experiment_protocol.json"

CONFIG_PATH.write_text(
    json.dumps(EXPERIMENT, indent=2),
    encoding="utf-8"
)

assert CONFIG_PATH.is_file()

print("Saved:", CONFIG_PATH)

# ------------------------------------------------
# 9. Final verification
# ------------------------------------------------

print("\n" + "=" * 76)
print("CELL 1 PASSED — EXPERIMENT PROTOCOL VERIFIED")
print("=" * 76)

print("Dataset: Official SLAKE English test")
print("Questions: 1061")
print("CLOSED: 416 | OPEN: 645")
print("FastV pruning after layer: 2")
print("Retentions: 75%, 50%, 25%")
print("Total required predictions: 3183")
print("Evaluation: EXACT B1 CELL 11B METRICS")
print("GPU count:", GPU_COUNT)
print("Configuration:", CONFIG_PATH)
print("=" * 76)


B2 — CELL 1: ENVIRONMENT AND EXPERIMENT VERIFICATION

[1] PROJECT DIRECTORIES
Project root: /kaggle/working/B2_FastV
Directories created: 9

[2] PYTHON AND PYTORCH
Python: 3.13.15
Python executable: /usr/bin/python3
PyTorch: 2.11.0+cu128
CUDA build: 12.8
CUDA available: True

[3] GPU HARDWARE
GPU count: 2

GPU 0
Name: Tesla T4
Total memory: 14.562 GiB
Compute capability: 7.5

GPU 1
Name: Tesla T4
Total memory: 14.562 GiB
Compute capability: 7.5

Two-GPU requirement: PASS

[4] DISCOVER SLAKE DATASET

Attached Kaggle inputs:
  /kaggle/input/datasets

SLAKE-format test files found: 1
/kaggle/input/datasets/aqilababy/slaek-data/Slake1.0/test.json | total: 2094 | English: 1061

Selected dataset root: /kaggle/input/datasets/aqilababy/slaek-data/Slake1.0
Official test JSON: /kaggle/input/datasets/aqilababy/slaek-data/Slake1.0/test.json
Image directory: /kaggle/input/datasets/aqilababy/slaek-data/Slake1.0/imgs

[5] OFFICIAL SLAKE SPLIT VERIFICATION
All-language test questions: 2094
English tes

In [3]:

# ================================================================
# B2 — CELL 2A: REPAIR PYTORCH DEPENDENCY CONFLICT
# ================================================================

import subprocess
import shutil
import json
import os
from pathlib import Path

ROOT = Path("/kaggle/working/B2_FastV")
PYTHON = ROOT / "venv/bin/python"
UV = shutil.which("uv")

assert PYTHON.is_file()
assert UV is not None

print("=" * 76)
print("B2 — CELL 2A: REPAIR AND LOCK PYTORCH")
print("=" * 76)

def run(args):
    print("\n$", " ".join(map(str, args)), flush=True)
    subprocess.run(
        [str(x) for x in args],
        check=True
    )

# ------------------------------------------------
# 1. Remove incompatible packages
# ------------------------------------------------

print("\n[1] REMOVE INCOMPATIBLE PYTORCH PACKAGES")

run([
    UV, "pip", "uninstall",
    "--python", PYTHON,
    "torch",
    "torchvision",
    "triton",
])

# ------------------------------------------------
# 2. Install matched CUDA 12.1 packages
# ------------------------------------------------

print("\n[2] INSTALL MATCHED PYTORCH + TORCHVISION")

run([
    UV, "pip", "install",
    "--python", PYTHON,
    "--index-url",
    "https://download.pytorch.org/whl/cu121",
    "torch==2.1.2",
    "torchvision==0.16.2",
])

# ------------------------------------------------
# 3. Check dependency consistency
# ------------------------------------------------

print("\n[3] CHECK INSTALLED PACKAGES")

run([
    UV, "pip", "check",
    "--python", PYTHON,
])

# ------------------------------------------------
# 4. Verify versions and GPU compatibility
# ------------------------------------------------

print("\n[4] VERIFY ENVIRONMENT")

verification = r"""
import sys
import torch
import torchvision
import transformers
import accelerate
import peft
import numpy
import timm
import sentencepiece
import safetensors

from transformers import MistralForCausalLM
from transformers.cache_utils import DynamicCache

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("NumPy:", numpy.__version__)
print("Timm:", timm.__version__)
print("GPU count:", torch.cuda.device_count())

assert sys.version_info[:2] == (3, 10)
assert torch.__version__ == "2.1.2+cu121"
assert torchvision.__version__ == "0.16.2+cu121"
assert torch.version.cuda == "12.1"
assert transformers.__version__ == "4.36.2"
assert accelerate.__version__ == "0.21.0"
assert peft.__version__ == "0.4.0"
assert numpy.__version__ == "1.26.4"

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2

for i in range(2):
    device = torch.device(f"cuda:{i}")
    x = torch.ones(4, device=device)
    y = x * 2

    assert y.sum().item() == 8.0

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i),
        "— CUDA computation PASS"
    )

print("Mistral import: PASS")
print("DynamicCache import: PASS")
print("ALL ENVIRONMENT CHECKS PASSED")
"""

run([PYTHON, "-c", verification])

# ------------------------------------------------
# 5. Record verified environment
# ------------------------------------------------

print("\n[5] SAVE ENVIRONMENT CONFIGURATION")

configuration = {
    "python": "3.10.21",
    "torch": "2.1.2+cu121",
    "torchvision": "0.16.2+cu121",
    "cuda_runtime": "12.1",
    "transformers": "4.36.2",
    "accelerate": "0.21.0",
    "peft": "0.4.0",
    "numpy": "1.26.4",
    "python_executable": str(PYTHON),
    "verification_passed": True,
    "experiment": "B2_FastV",
    "retentions": [75, 50, 25],
}

config_path = ROOT / "configs/environment.json"

config_path.write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8",
)

print("Saved:", config_path)

print("\n" + "=" * 76)
print("CELL 2A PASSED — PYTORCH ENVIRONMENT REPAIRED")
print("=" * 76)


B2 — CELL 2A: REPAIR AND LOCK PYTORCH

[1] REMOVE INCOMPATIBLE PYTORCH PACKAGES

$ /usr/local/bin/uv pip uninstall --python /kaggle/working/B2_FastV/venv/bin/python torch torchvision triton


Using Python 3.10.21 environment at: B2_FastV/venv



[2] INSTALL MATCHED PYTORCH + TORCHVISION

$ /usr/local/bin/uv pip install --python /kaggle/working/B2_FastV/venv/bin/python --index-url https://download.pytorch.org/whl/cu121 torch==2.1.2 torchvision==0.16.2


Uninstalled 3 packages in 454ms
 - torch==2.14.1
 - torchvision==0.29.1
 - triton==3.8.0
Using Python 3.10.21 environment at: B2_FastV/venv
Resolved 18 packages in 503ms
Prepared 1 package in 95ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.



[3] CHECK INSTALLED PACKAGES

$ /usr/local/bin/uv pip check --python /kaggle/working/B2_FastV/venv/bin/python

[4] VERIFY ENVIRONMENT

$ /kaggle/working/B2_FastV/venv/bin/python -c 
import sys
import torch
import torchvision
import transformers
import accelerate
import peft
import numpy
import timm
import sentencepiece
import safetensors

from transformers import MistralForCausalLM
from transformers.cache_utils import DynamicCache

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("NumPy:", numpy.__version__)
print("Timm:", timm.__version__)
print("GPU count:", torch.cuda.device_count())

assert sys.version_info[:2] == (3, 10)
assert torch.__version__ == "2.1.2+cu121"
assert torchvision.__version__ == "0.16.2+cu121"
assert torch.version.cuda == "

Installed 3 packages in 7.41s
 + torch==2.1.2+cu121
 + torchvision==0.16.2+cu121
 + triton==2.1.0
Using Python 3.10.21 environment at: B2_FastV/venv
Checked 70 packages in 1ms
All installed packages are compatible


Python: 3.10.21
PyTorch: 2.1.2+cu121
Torchvision: 0.16.2+cu121
CUDA: 12.1
Transformers: 4.36.2
Accelerate: 0.21.0
PEFT: 0.4.0
NumPy: 1.26.4
Timm: 1.0.20
GPU count: 2
GPU 0: Tesla T4 — CUDA computation PASS
GPU 1: Tesla T4 — CUDA computation PASS
Mistral import: PASS
DynamicCache import: PASS
ALL ENVIRONMENT CHECKS PASSED

[5] SAVE ENVIRONMENT CONFIGURATION
Saved: /kaggle/working/B2_FastV/configs/environment.json

CELL 2A PASSED — PYTORCH ENVIRONMENT REPAIRED


In [4]:

# ================================================================
# B2 — CELL 3: DOWNLOAD & VERIFY MODEL CHECKPOINT
# Frozen LLaVA-Med v1.5 Mistral-7B
# ================================================================

from pathlib import Path
import subprocess
import json
import shutil
import os

ROOT = Path("/kaggle/working/B2_FastV")
PYTHON = ROOT / "venv/bin/python"
CONFIG_DIR = ROOT / "configs"
SCRIPTS_DIR = ROOT / "scripts"

MODEL_ID = "microsoft/llava-med-v1.5-mistral-7b"
REVISION = "91bb16c122001ddc9cf1fd36ce1dae09448943a2"

CACHE_DIR = Path("/tmp/b2_llava_med_hf_cache")
REPORT = CONFIG_DIR / "model_checkpoint.json"
SCRIPT = SCRIPTS_DIR / "cell3_download_checkpoint.py"

assert PYTHON.is_file(), "Cell 2A must pass first."
assert (CONFIG_DIR / "environment.json").is_file()

SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("B2 — CELL 3: OFFICIAL LLAVA-MED CHECKPOINT")
print("=" * 76)

print("\nModel:", MODEL_ID)
print("Revision:", REVISION)
print("Cache:", CACHE_DIR)

# ------------------------------------------------
# 1. Write standalone Python 3.10 download script
# ------------------------------------------------

source = r'''
import os
import json
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download
from safetensors import safe_open

ROOT = Path("/kaggle/working/B2_FastV")
CACHE_DIR = Path("/tmp/b2_llava_med_hf_cache")

MODEL_ID = "microsoft/llava-med-v1.5-mistral-7b"
REVISION = "91bb16c122001ddc9cf1fd36ce1dae09448943a2"

REPORT = ROOT / "configs/model_checkpoint.json"

print("=" * 76, flush=True)
print("B2 — MODEL DOWNLOAD AND VERIFICATION", flush=True)
print("=" * 76, flush=True)

# ------------------------------------------------
# 1. Check available storage
# ------------------------------------------------

print("\n[1] DISK SPACE", flush=True)

CACHE_DIR.mkdir(parents=True, exist_ok=True)

disk = shutil.disk_usage(CACHE_DIR)

print(
    "Available space:",
    round(disk.free / 1024**3, 2),
    "GiB",
    flush=True,
)

# This is a warning, not an unconditional failure:
# an existing checkpoint may already be cached.
if disk.free < 17 * 1024**3:
    print(
        "WARNING: Less than 17 GiB free. "
        "A fresh model download may fail.",
        flush=True,
    )

# ------------------------------------------------
# 2. Download exact revision
# ------------------------------------------------

print("\n[2] RESOLVE PINNED MODEL SNAPSHOT", flush=True)

snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=REVISION,
    cache_dir=str(CACHE_DIR),
    allow_patterns=[
        "*.json",
        "*.safetensors",
        "*.model",
        "tokenizer*",
        "special_tokens_map.json",
        "added_tokens.json",
    ],
    max_workers=4,
    resume_download=True,
)

snapshot = Path(snapshot_path)

print("Snapshot:", snapshot, flush=True)

assert snapshot.is_dir()
assert snapshot.name == REVISION, (
    "Unexpected model revision."
)

# ------------------------------------------------
# 3. Verify checkpoint index
# ------------------------------------------------

print("\n[3] CHECK CHECKPOINT INDEX", flush=True)

config_path = snapshot / "config.json"
index_path = (
    snapshot / "model.safetensors.index.json"
)

assert config_path.is_file(), "Missing config.json"
assert index_path.is_file(), "Missing weight index"

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

index = json.loads(
    index_path.read_text(encoding="utf-8")
)

print("Model type:", config.get("model_type"), flush=True)
print(
    "Decoder layers:",
    config.get("num_hidden_layers"),
    flush=True,
)

assert config.get("model_type") == "llava_mistral"
assert config.get("num_hidden_layers") == 32

weight_map = index["weight_map"]

assert isinstance(weight_map, dict)
assert weight_map

shard_names = sorted(set(weight_map.values()))

expected_shards = [
    f"model-{i:05d}-of-00004.safetensors"
    for i in range(1, 5)
]

assert shard_names == expected_shards, (
    f"Unexpected checkpoint shards: {shard_names}"
)

assert len(weight_map) == 686, (
    f"Unexpected tensor count: {len(weight_map)}"
)

print("Indexed tensors:", len(weight_map), flush=True)
print("Expected shards:", len(shard_names), flush=True)

# ------------------------------------------------
# 4. Verify all weight shards
# ------------------------------------------------

print("\n[4] VERIFY SAFETENSORS FILES", flush=True)

total_bytes = 0
verified_keys = set()

for name in shard_names:
    shard = snapshot / name

    assert shard.is_file(), (
        f"Missing weight shard: {name}"
    )

    size = shard.stat().st_size
    total_bytes += size

    # safe_open verifies that the container can be
    # parsed and exposes its indexed tensors.
    with safe_open(
        str(shard),
        framework="pt",
        device="cpu",
    ) as handle:
        actual_keys = set(handle.keys())

    expected_keys = {
        key
        for key, mapped_shard in weight_map.items()
        if mapped_shard == name
    }

    assert actual_keys == expected_keys, (
        f"Tensor index mismatch in {name}"
    )

    assert not (
        verified_keys & actual_keys
    ), "Duplicate tensor names across shards"

    verified_keys.update(actual_keys)

    print(
        name,
        "|",
        round(size / 1024**3, 3),
        "GiB",
        "|",
        len(actual_keys),
        "tensors",
        flush=True,
    )

assert verified_keys == set(weight_map)

expected_bytes = 15132534160

assert total_bytes == expected_bytes, (
    f"Unexpected checkpoint size: {total_bytes}"
)

print(
    "\nTotal weight size:",
    round(total_bytes / 1024**3, 3),
    "GiB",
    flush=True,
)

print("Checkpoint index consistency: PASS", flush=True)

# ------------------------------------------------
# 5. Verify key model components
# ------------------------------------------------

print("\n[5] VERIFY MODEL COMPONENTS", flush=True)

vision_keys = [
    k for k in weight_map
    if k.startswith(
        "model.vision_tower."
    )
]

projector_keys = [
    k for k in weight_map
    if k.startswith(
        "model.mm_projector."
    )
]

decoder_layers = set()

import re

for key in weight_map:
    match = re.match(
        r"model\.layers\.(\d+)\.",
        key
    )
    if match:
        decoder_layers.add(int(match.group(1)))

assert len(vision_keys) == 391
assert len(projector_keys) == 4
assert decoder_layers == set(range(32))

print("Vision tensors:", len(vision_keys), flush=True)
print("Projector tensors:", len(projector_keys), flush=True)
print("Decoder layers:", len(decoder_layers), flush=True)

assert (snapshot / "tokenizer.model").is_file(), (
    "Missing Mistral tokenizer.model"
)

print("Tokenizer files: PASS", flush=True)

# ------------------------------------------------
# 6. Save verified checkpoint metadata
# ------------------------------------------------

print("\n[6] SAVE CHECKPOINT CONFIGURATION", flush=True)

report = {
    "model_id": MODEL_ID,
    "revision": REVISION,
    "snapshot_path": str(snapshot),
    "cache_dir": str(CACHE_DIR),
    "model_type": config["model_type"],
    "num_hidden_layers": 32,
    "hidden_size": config.get("hidden_size"),
    "vision_tower": (
        "openai/clip-vit-large-patch14-336"
    ),
    "shard_names": shard_names,
    "shard_count": len(shard_names),
    "total_weight_bytes": total_bytes,
    "indexed_tensor_count": len(weight_map),
    "vision_tower_tensor_count": len(vision_keys),
    "projector_tensor_count": len(projector_keys),
    "checkpoint_verified": True,
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:", REPORT, flush=True)

print("\n" + "=" * 76, flush=True)
print(
    "CELL 3 PASSED — ORIGINAL CHECKPOINT VERIFIED",
    flush=True,
)
print("=" * 76, flush=True)
'''

SCRIPT.write_text(source, encoding="utf-8")

print("\n[1] CREATED DOWNLOAD SCRIPT")
print(SCRIPT)

# ------------------------------------------------
# 2. Syntax check
# ------------------------------------------------

print("\n[2] VERIFY SCRIPT SYNTAX")

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("Syntax: PASS")

# ------------------------------------------------
# 3. Download and verify
# ------------------------------------------------

print("\n[3] EXECUTE CHECKPOINT DOWNLOAD\n", flush=True)

env = os.environ.copy()
env["HF_HOME"] = str(CACHE_DIR)
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(SCRIPT)],
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 3 checkpoint download or verification failed. "
        "Share the full error before proceeding."
    )

assert REPORT.is_file()

checkpoint = json.loads(
    REPORT.read_text(encoding="utf-8")
)

assert checkpoint["checkpoint_verified"] is True

print("\n" + "=" * 76)
print("B2 — CELL 3 COMPLETE")
print("=" * 76)
print("Checkpoint:", checkpoint["snapshot_path"])
print("Shards:", checkpoint["shard_count"])
print("Indexed tensors:", checkpoint["indexed_tensor_count"])
print("Checkpoint integrity: PASS")


B2 — CELL 3: OFFICIAL LLAVA-MED CHECKPOINT

Model: microsoft/llava-med-v1.5-mistral-7b
Revision: 91bb16c122001ddc9cf1fd36ce1dae09448943a2
Cache: /tmp/b2_llava_med_hf_cache

[1] CREATED DOWNLOAD SCRIPT
/kaggle/working/B2_FastV/scripts/cell3_download_checkpoint.py

[2] VERIFY SCRIPT SYNTAX
Syntax: PASS

[3] EXECUTE CHECKPOINT DOWNLOAD

B2 — MODEL DOWNLOAD AND VERIFICATION

[1] DISK SPACE
Available space: 1060.4 GiB

[2] RESOLVE PINNED MODEL SNAPSHOT


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]
config.json: 0.00B [00:00, ?B/s]

generation_config.json: 100%|██████████| 111/111 [00:00<00:00, 479kB/s]
config.json: 1.41kB [00:00, 777kB/s]
Fetching 10 files:  10%|█         | 1/10 [00:00<00:02,  3.78it/s]
model-00001-of-00004.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]


model-00003-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]



model-00004-of-00004.safetensors:   0%|          | 0.00/262M [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 10.5M/5.00G [00:01<08:29, 9.79MB/s]



model-00004-of-00004.safetensors:   4%|▍         | 10.5M/262M [00:01<00:25, 9.73MB/s]


model-00003-of-00004.safetensors:   0%|          | 10.5M/4.93G [00:01<08:32, 9.60MB/s]
model-00001-of-00004.safetensors:   0%|          | 10.5M/4.94G [00:01<11:04, 7.43MB/s]

model-00002-of-00004.safetensors:   1%|          | 31.5M/5.00G

Snapshot: /tmp/b2_llava_med_hf_cache/models--microsoft--llava-med-v1.5-mistral-7b/snapshots/91bb16c122001ddc9cf1fd36ce1dae09448943a2

[3] CHECK CHECKPOINT INDEX
Model type: llava_mistral
Decoder layers: 32
Indexed tensors: 686
Expected shards: 4

[4] VERIFY SAFETENSORS FILES
model-00001-of-00004.safetensors | 4.604 GiB | 97 tensors
model-00002-of-00004.safetensors | 4.656 GiB | 106 tensors
model-00003-of-00004.safetensors | 4.589 GiB | 482 tensors
model-00004-of-00004.safetensors | 0.244 GiB | 1 tensors

Total weight size: 14.093 GiB
Checkpoint index consistency: PASS

[5] VERIFY MODEL COMPONENTS
Vision tensors: 391
Projector tensors: 4
Decoder layers: 32
Tokenizer files: PASS

[6] SAVE CHECKPOINT CONFIGURATION
Saved: /kaggle/working/B2_FastV/configs/model_checkpoint.json

CELL 3 PASSED — ORIGINAL CHECKPOINT VERIFIED

B2 — CELL 3 COMPLETE
Checkpoint: /tmp/b2_llava_med_hf_cache/models--microsoft--llava-med-v1.5-mistral-7b/snapshots/91bb16c122001ddc9cf1fd36ce1dae09448943a2
Shards: 4
Inde

In [5]:

# ================================================================
# B2 — CELL 4: PINNED REPOSITORY + FROZEN TWO-GPU LOADER
# ================================================================

from pathlib import Path
import subprocess
import json
import os
import textwrap

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"
LOADER = ROOT / "load_b2_model.py"
TEST = ROOT / "scripts/cell4_verify_loader.py"
CONFIG = ROOT / "configs/model_checkpoint.json"

REPO_URL = "https://github.com/microsoft/LLaVA-Med.git"
REPO_COMMIT = "30697ca50b5c29a8e955c99330b259776aef27b9"

assert PYTHON.is_file()
assert CONFIG.is_file()

checkpoint = json.loads(CONFIG.read_text(encoding="utf-8"))
SNAPSHOT = Path(checkpoint["snapshot_path"])

assert checkpoint["checkpoint_verified"] is True
assert SNAPSHOT.is_dir()

for name in checkpoint["shard_names"]:
    assert (SNAPSHOT / name).is_file(), name

print("=" * 76)
print("B2 — CELL 4: PINNED LLAVA-MED MODEL LOADER")
print("=" * 76)

# ------------------------------------------------
# 1. Clone and verify source revision
# ------------------------------------------------

print("\n[1] VERIFY SOURCE REPOSITORY", flush=True)

if not (REPO / ".git").is_dir():
    assert not REPO.exists() or not any(REPO.iterdir()), (
        "Repository path exists but is not a Git repository."
    )

    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", REPO_COMMIT],
    check=True
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == REPO_COMMIT

for relative in (
    "llava/__init__.py",
    "llava/model/language_model/llava_mistral.py",
    "llava/model/multimodal_encoder/clip_encoder.py",
):
    assert (REPO / relative).is_file(), relative

print("Repository commit:", actual_commit)
print("Pinned repository: PASS")

# ------------------------------------------------
# 2. Create the exact-checkpoint model loader
# ------------------------------------------------

print("\n[2] CREATE MODEL LOADER", flush=True)

loader_code = r'''
import sys
import json
from pathlib import Path

import torch
from safetensors import safe_open

from transformers import (
    AutoTokenizer,
    CLIPVisionModel,
    CLIPVisionConfig,
    CLIPImageProcessor,
)

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from llava.model.language_model.llava_mistral import (
    LlavaMistralForCausalLM,
)
from llava.model.multimodal_encoder.clip_encoder import (
    CLIPVisionTower,
)

MODEL_CONFIG = ROOT / "configs/model_checkpoint.json"


def install_checkpoint_vision_patch():
    """
    Build CLIP's architecture before loading the combined
    LLaVA-Med checkpoint.

    This avoids downloading a second set of pretrained
    CLIP weights and ensures the checkpoint supplies
    the actual vision encoder parameters.
    """
    if getattr(CLIPVisionTower, "_b2_patched", False):
        return

    def load_model_from_checkpoint(self):
        if self.is_loaded:
            return

        vision_config = CLIPVisionConfig(
            hidden_size=1024,
            intermediate_size=4096,
            num_hidden_layers=24,
            num_attention_heads=16,
            image_size=336,
            patch_size=14,
            num_channels=3,
            attention_dropout=0.0,
            layer_norm_eps=1e-5,
        )

        self.vision_tower = CLIPVisionModel(vision_config)
        self.vision_tower.requires_grad_(False)
        self.is_loaded = True

    CLIPVisionTower.load_model = load_model_from_checkpoint

    # Original LLaVA delays vision tower initialization.
    # The combined checkpoint includes vision weights,
    # so the parameters must exist during from_pretrained.
    original_init = CLIPVisionTower.__init__

    def eager_init(self, vision_tower, args, delay_load=False):
        return original_init(
            self, vision_tower, args, delay_load=False
        )

    CLIPVisionTower.__init__ = eager_init
    CLIPVisionTower._b2_patched = True


def load_b2_model():
    """
    Return tokenizer, frozen model, vision tower.

    GPU 0:
        vision encoder, projector, token embeddings,
        decoder layers 0-15

    GPU 1:
        decoder layers 16-31, norm, LM head
    """
    config = json.loads(
        MODEL_CONFIG.read_text(encoding="utf-8")
    )
    snapshot = Path(config["snapshot_path"])

    assert snapshot.is_dir()
    assert torch.cuda.device_count() >= 2

    install_checkpoint_vision_patch()

    print("=" * 72, flush=True)
    print("B2 — FROZEN TWO-GPU LLAVA-MED LOAD", flush=True)
    print("=" * 72, flush=True)

    print("\n[1] TOKENIZER", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(
        str(snapshot),
        use_fast=False,
        local_files_only=True,
    )

    assert len(tokenizer) == 32000
    print("Vocabulary:", len(tokenizer), flush=True)

    print("\n[2] LOAD ORIGINAL CHECKPOINT", flush=True)

    device_map = {
        "model.embed_tokens": 0,
        "model.vision_tower": 0,
        "model.mm_projector": 0,
        "model.norm": 1,
        "lm_head": 1,
    }

    for layer in range(32):
        device_map[f"model.layers.{layer}"] = (
            0 if layer < 16 else 1
        )

    model, loading_info = (
        LlavaMistralForCausalLM.from_pretrained(
            str(snapshot),
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map=device_map,
            attn_implementation="eager",
            local_files_only=True,
            output_loading_info=True,
        )
    )

    for name in (
        "missing_keys",
        "unexpected_keys",
        "mismatched_keys",
        "error_msgs",
    ):
        values = loading_info.get(name, [])
        print(f"{name}: {len(values)}", flush=True)

        if values:
            print(values[:10], flush=True)

        assert not values, f"Checkpoint issue: {name}"

    assert model.config.num_hidden_layers == 32

    print("\n[3] VERIFY GPU PLACEMENT", flush=True)

    modules = dict(model.named_modules())

    placements = {
        "model.embed_tokens": 0,
        "model.vision_tower": 0,
        "model.mm_projector": 0,
        "model.layers.0": 0,
        "model.layers.15": 0,
        "model.layers.16": 1,
        "model.layers.31": 1,
        "model.norm": 1,
        "lm_head": 1,
    }

    for name, expected in placements.items():
        assert name in modules, name

        parameter = next(modules[name].parameters())
        device = parameter.device

        assert device.type == "cuda"
        assert device.index == expected, (
            name, str(device), expected
        )

        print(f"{name}: {device} — PASS", flush=True)

    print("\n[4] VERIFY VISION WEIGHTS", flush=True)

    index_path = (
        snapshot / "model.safetensors.index.json"
    )
    weight_map = json.loads(
        index_path.read_text(encoding="utf-8")
    )["weight_map"]

    key = (
        "model.vision_tower.vision_tower."
        "vision_model.embeddings.patch_embedding.weight"
    )

    assert key in weight_map

    shard_path = snapshot / weight_map[key]

    with safe_open(
        str(shard_path),
        framework="pt",
        device="cpu",
    ) as reader:
        reference_weight = reader.get_tensor(key)

    actual_weight = dict(model.named_parameters())[
        key
    ].detach().cpu()

    assert reference_weight.shape == actual_weight.shape

    maximum_error = (
        reference_weight.float() -
        actual_weight.float()
    ).abs().max().item()

    print(
        "Vision weight max absolute error:",
        maximum_error,
        flush=True,
    )
    assert maximum_error == 0.0

    print("Original vision weights: EXACT MATCH", flush=True)

    print("\n[5] FREEZE MODEL", flush=True)

    model.eval()
    model.requires_grad_(False)

    parameter_count = sum(
        p.numel() for p in model.parameters()
    )
    trainable_count = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    assert parameter_count == 7566219264, (
        f"Unexpected parameter count: {parameter_count}"
    )
    assert trainable_count == 0
    assert not any(p.is_meta for p in model.parameters())

    print("Parameters:", parameter_count, flush=True)
    print("Trainable:", trainable_count, flush=True)

    print("\n[6] INITIALIZE IMAGE PROCESSOR", flush=True)

    vision_tower = model.get_vision_tower()
    assert vision_tower is not None

    # Use the canonical preprocessing configuration.
    # No separate CLIP model weights are downloaded.
    vision_tower.image_processor = CLIPImageProcessor(
        do_resize=True,
        size={"shortest_edge": 336},
        do_center_crop=True,
        crop_size={"height": 336, "width": 336},
        do_rescale=True,
        rescale_factor=1.0 / 255.0,
        do_normalize=True,
        image_mean=[0.48145466, 0.4578275, 0.40821073],
        image_std=[0.26862954, 0.26130258, 0.27577711],
    )

    assert vision_tower.num_patches == 576

    print("Vision patches: 576 — PASS", flush=True)

    print("\n[7] GPU MEMORY", flush=True)

    for gpu in (0, 1):
        print(
            f"GPU {gpu}: "
            f"{torch.cuda.memory_allocated(gpu)/1024**3:.3f} "
            "GiB allocated",
            flush=True,
        )

    report = {
        "model_id": config["model_id"],
        "revision": config["revision"],
        "source_revision": (
            "30697ca50b5c29a8e955c99330b259776aef27b9"
        ),
        "parameter_count": parameter_count,
        "trainable_count": trainable_count,
        "vision_weight_max_error": maximum_error,
        "decoder_layers": 32,
        "vision_tokens": 576,
        "device_map": device_map,
        "verified": True,
    }

    report_path = (
        ROOT / "configs/model_load_report.json"
    )
    report_path.write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    print("\nSaved:", report_path, flush=True)

    print("\n" + "=" * 72, flush=True)
    print("CELL 4 MODEL VERIFICATION PASSED", flush=True)
    print("=" * 72, flush=True)

    return tokenizer, model, vision_tower
'''

LOADER.write_text(
    textwrap.dedent(loader_code).lstrip(),
    encoding="utf-8",
)

assert LOADER.is_file()
print("Created:", LOADER)

# ------------------------------------------------
# 3. Create verification script before executing
# ------------------------------------------------

print("\n[3] CREATE MODEL VERIFICATION SCRIPT", flush=True)

test_code = r'''
import sys

sys.path.insert(
    0, "/kaggle/working/B2_FastV"
)
sys.path.insert(
    0, "/kaggle/working/LLaVA-Med"
)

from load_b2_model import load_b2_model

tokenizer, model, vision_tower = load_b2_model()

assert tokenizer is not None
assert model is not None
assert vision_tower is not None

print("\nB2 LOADER VERIFICATION: PASS", flush=True)
'''

TEST.parent.mkdir(parents=True, exist_ok=True)
TEST.write_text(
    textwrap.dedent(test_code).lstrip(),
    encoding="utf-8",
)

assert TEST.is_file()
print("Created:", TEST)

# ------------------------------------------------
# 4. Check script syntax
# ------------------------------------------------

print("\n[4] VERIFY PYTHON SYNTAX", flush=True)

for script in (LOADER, TEST):
    subprocess.run(
        [str(PYTHON), "-m", "py_compile", str(script)],
        check=True,
    )
    print("Syntax PASS:", script.name)

# ------------------------------------------------
# 5. Run independent model verification
# ------------------------------------------------

print("\n[5] LOAD AND VERIFY FROZEN MODEL\n", flush=True)

env = os.environ.copy()
env["HF_HOME"] = checkpoint["cache_dir"]
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])

result = subprocess.run(
    [str(PYTHON), "-u", str(TEST)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 4 model verification failed. "
        "Do not proceed until the traceback is resolved."
    )

assert (
    ROOT / "configs/model_load_report.json"
).is_file()

print("\n" + "=" * 76)
print("B2 — CELL 4 PASSED")
print("=" * 76)
print("Repository: exact pinned revision")
print("Checkpoint: exact original LLaVA-Med weights")
print("Model: frozen")
print("GPU distribution: verified")
print("FastV implementation: not yet applied")
print("Experimental retentions: 75%, 50%, 25%")


B2 — CELL 4: PINNED LLAVA-MED MODEL LOADER

[1] VERIFY SOURCE REPOSITORY


Cloning into '/kaggle/working/LLaVA-Med'...


Repository commit: 30697ca50b5c29a8e955c99330b259776aef27b9
Pinned repository: PASS

[2] CREATE MODEL LOADER
Created: /kaggle/working/B2_FastV/load_b2_model.py

[3] CREATE MODEL VERIFICATION SCRIPT
Created: /kaggle/working/B2_FastV/scripts/cell4_verify_loader.py

[4] VERIFY PYTHON SYNTAX
Syntax PASS: load_b2_model.py
Syntax PASS: cell4_verify_loader.py

[5] LOAD AND VERIFY FROZEN MODEL



HEAD is now at 30697ca Adds data to data directory


B2 — FROZEN TWO-GPU LLAVA-MED LOAD

[1] TOKENIZER
Vocabulary: 32000

[2] LOAD ORIGINAL CHECKPOINT


Loading checkpoint shards: 100%|██████████| 4/4 [01:03<00:00, 15.79s/it]


missing_keys: 0
unexpected_keys: 0
mismatched_keys: 0
error_msgs: 0

[3] VERIFY GPU PLACEMENT
model.embed_tokens: cuda:0 — PASS
model.vision_tower: cuda:0 — PASS
model.mm_projector: cuda:0 — PASS
model.layers.0: cuda:0 — PASS
model.layers.15: cuda:0 — PASS
model.layers.16: cuda:1 — PASS
model.layers.31: cuda:1 — PASS
model.norm: cuda:1 — PASS
lm_head: cuda:1 — PASS

[4] VERIFY VISION WEIGHTS
Vision weight max absolute error: 0.0
Original vision weights: EXACT MATCH

[5] FREEZE MODEL
Parameters: 7566219264
Trainable: 0

[6] INITIALIZE IMAGE PROCESSOR
Vision patches: 576 — PASS

[7] GPU MEMORY
GPU 0: 7.599 GiB allocated
GPU 1: 6.994 GiB allocated

Saved: /kaggle/working/B2_FastV/configs/model_load_report.json

CELL 4 MODEL VERIFICATION PASSED

B2 LOADER VERIFICATION: PASS

B2 — CELL 4 PASSED
Repository: exact pinned revision
Checkpoint: exact original LLaVA-Med weights
Model: frozen
GPU distribution: verified
FastV implementation: not yet applied
Experimental retentions: 75%, 50%, 25%


In [7]:

# ================================================================
# B2 — CELL 5: CORRECT ATTENTION ARCHITECTURE AUDIT
# ================================================================

from pathlib import Path
import subprocess
import os

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

SCRIPT = ROOT / "scripts/cell5_fastv_architecture_audit.py"
REPORT = ROOT / "configs/fastv_architecture_audit.json"

print("=" * 76)
print("B2 — CELL 5A: CORRECT ATTENTION AUDIT")
print("=" * 76)

assert SCRIPT.is_file(), "Original Cell 5 script missing."
assert PYTHON.is_file()

source = SCRIPT.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Correct softmax implementation detection
# ------------------------------------------------------------

print("\n[1] CORRECT ATTENTION SOFTMAX CHECK")

old = 'assert "torch.softmax" in attention_source'

new = '''assert (
    "softmax(" in attention_source
), "Attention implementation has no visible softmax operation"'''

assert source.count(old) == 1, (
    "Expected exactly one original softmax assertion."
)

source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 2. Correct checkpoint configuration loading
# ------------------------------------------------------------

print("\n[2] CORRECT MODEL CONFIGURATION LOADING")

old_import = "from transformers import MistralConfig"

new_import = "from transformers import AutoConfig"

assert source.count(old_import) == 1
source = source.replace(old_import, new_import, 1)

old_config = '''config = MistralConfig.from_pretrained(
    str(snapshot),
    local_files_only=True,
)'''

new_config = '''config = AutoConfig.from_pretrained(
    str(snapshot),
    local_files_only=True,
)'''

assert source.count(old_config) == 1
source = source.replace(old_config, new_config, 1)

# ------------------------------------------------------------
# 3. Save corrected script
# ------------------------------------------------------------

print("\n[3] SAVE CORRECTED AUDIT")

SCRIPT.write_text(source, encoding="utf-8")

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("Syntax: PASS")

# ------------------------------------------------------------
# 4. Rerun architecture audit
# ------------------------------------------------------------

print("\n[4] RERUN ARCHITECTURE AUDIT\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])

result = subprocess.run(
    [str(PYTHON), "-u", str(SCRIPT)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 5A audit failed. "
        "Share the traceback before proceeding."
    )

assert REPORT.is_file()

print("\n" + "=" * 76)
print("CELL 5A PASSED — FASTV ARCHITECTURE VERIFIED")
print("=" * 76)


B2 — CELL 5A: CORRECT ATTENTION AUDIT

[1] CORRECT ATTENTION SOFTMAX CHECK

[2] CORRECT MODEL CONFIGURATION LOADING

[3] SAVE CORRECTED AUDIT
Syntax: PASS

[4] RERUN ARCHITECTURE AUDIT

B2 — FASTV ARCHITECTURE INSPECTION

[1] LIBRARY VERSIONS
PyTorch: 2.1.2+cu121
Transformers: 4.36.2

[2] ORIGINAL MODEL ARCHITECTURE
Decoder layers: 32
Hidden size: 4096
Attention heads: 32
KV heads: 8
Head dimension: 128
Sliding window: None
Vision patches: 576

[3] DECODER LAYER FORWARD API
Signature: (self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, position_ids: Optional[torch.LongTensor] = None, past_key_value: Optional[Tuple[torch.Tensor]] = None, output_attentions: Optional[bool] = False, use_cache: Optional[bool] = False, **kwargs) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]
Required arguments: PASS

[4] ATTENTION EXTRACTION
Attention API: (self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, posi

In [8]:

# ================================================================
# B2 — CELL 6: FASTV ATTENTION-GUIDED TOKEN SELECTION
#
# Creates:
#   fastv_selection.py
#   scripts/cell6_test_fastv_selection.py
#   configs/fastv_selection_test.json
#
# Does not load the 7B model.
# ================================================================

from pathlib import Path
import subprocess
import os
import textwrap

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

MODULE = ROOT / "fastv_selection.py"
TEST = ROOT / "scripts/cell6_test_fastv_selection.py"
REPORT = ROOT / "configs/fastv_selection_test.json"

assert PYTHON.is_file()
assert (
    ROOT / "configs/fastv_architecture_audit.json"
).is_file()

print("=" * 76)
print("B2 — CELL 6: FASTV TOKEN SELECTION")
print("=" * 76)

# ------------------------------------------------
# 1. Write production selection module
# ------------------------------------------------

module_code = r'''
import torch

VISUAL_TOKEN_COUNT = 576
RETENTION_TO_COUNT = {
    75: 432,
    50: 288,
    25: 144,
}


def compute_visual_importance(
    attention_weights,
    visual_start,
    visual_count=VISUAL_TOKEN_COUNT,
):
    """
    FastV visual-token importance.

    attention_weights:
        [batch, heads, queries, keys]

    Scores:
        Mean attention RECEIVED by each visual key,
        averaged across attention heads and query positions.

    Assumes the supplied attention weights already
    include the model's causal masking.
    """
    if attention_weights.ndim != 4:
        raise ValueError(
            "Attention must have shape [B,H,Q,K]."
        )

    batch, heads, queries, keys = (
        attention_weights.shape
    )

    if batch < 1 or heads < 1 or queries < 1:
        raise ValueError("Invalid attention shape.")

    if not (0 <= visual_start <= keys - visual_count):
        raise ValueError("Invalid visual token span.")

    visual_attention = attention_weights[
        ...,
        visual_start:visual_start + visual_count
    ]

    scores = visual_attention.float().mean(
        dim=(1, 2)
    )

    if scores.shape != (batch, visual_count):
        raise AssertionError("Importance shape mismatch.")

    if not bool(torch.isfinite(scores).all()):
        raise ValueError("Nonfinite attention scores.")

    return scores


def select_fastv_indices(
    attention_weights,
    visual_start,
    retention_pct,
    visual_count=VISUAL_TOKEN_COUNT,
):
    """
    Return the retained positions in original token order.

    Only visual tokens can be removed.
    All text tokens before and after the visual span
    must remain intact.

    Output:
        keep_indices [batch, retained_seq_len]
        importance [batch, visual_count]
    """
    if retention_pct not in RETENTION_TO_COUNT:
        raise ValueError(
            "Experimental B2 retention must be "
            "75%, 50%, or 25%."
        )

    if visual_count != VISUAL_TOKEN_COUNT:
        raise ValueError("Expected exactly 576 visual tokens.")

    if attention_weights.ndim != 4:
        raise ValueError("Expected rank-four attention.")

    batch, _, queries, keys = attention_weights.shape

    if queries != keys:
        raise ValueError(
            "FastV prefill requires square attention."
        )

    keep_count = RETENTION_TO_COUNT[retention_pct]

    importance = compute_visual_importance(
        attention_weights,
        visual_start,
        visual_count,
    )

    selected_batches = []

    for b in range(batch):
        # Stable sorting resolves ties reproducibly.
        ranked = torch.argsort(
            importance[b],
            descending=True,
            stable=True,
        )

        relative_visual_indices = (
            ranked[:keep_count].sort().values
        )

        absolute_visual_indices = (
            relative_visual_indices + visual_start
        )

        prefix = torch.arange(
            0,
            visual_start,
            device=attention_weights.device,
        )

        suffix = torch.arange(
            visual_start + visual_count,
            keys,
            device=attention_weights.device,
        )

        indices = torch.cat([
            prefix,
            absolute_visual_indices,
            suffix,
        ])

        selected_batches.append(indices)

    keep_indices = torch.stack(selected_batches)

    expected_length = keys - visual_count + keep_count

    assert keep_indices.shape == (
        batch,
        expected_length,
    )

    assert bool(torch.all(
        keep_indices[:, 1:] >
        keep_indices[:, :-1]
    ))

    return keep_indices, importance


def gather_sequence(tensor, keep_indices):
    """
    Physically remove unselected sequence positions.

    Supports:
      [batch, sequence]
      [batch, sequence, hidden]
    """
    if keep_indices.ndim != 2:
        raise ValueError("Indices must be rank two.")

    if tensor.shape[0] != keep_indices.shape[0]:
        raise ValueError("Batch dimension mismatch.")

    indices = keep_indices.to(tensor.device)

    if tensor.ndim == 2:
        return tensor.gather(dim=1, index=indices)

    if tensor.ndim == 3:
        expanded = indices.unsqueeze(-1).expand(
            -1,
            -1,
            tensor.shape[-1],
        )
        return tensor.gather(dim=1, index=expanded)

    raise ValueError("Unsupported tensor rank.")


def gather_causal_attention_mask(
    attention_mask,
    keep_indices,
):
    """
    Reduce the query and key axes together.

    Input:
        [batch, 1, original_seq, original_seq]

    Output:
        [batch, 1, retained_seq, retained_seq]
    """
    if attention_mask.ndim != 4:
        raise ValueError("Expected a 4D causal mask.")

    batch, heads, query_len, key_len = (
        attention_mask.shape
    )

    if heads != 1 or query_len != key_len:
        raise ValueError(
            "Expected square prefill causal mask."
        )

    if keep_indices.shape[0] != batch:
        raise ValueError("Batch size mismatch.")

    result = []

    for b in range(batch):
        index = keep_indices[b].to(
            attention_mask.device
        )

        reduced = attention_mask[b:b+1].index_select(
            -2, index
        ).index_select(
            -1, index
        )

        result.append(reduced)

    return torch.cat(result, dim=0)
'''

MODULE.write_text(
    textwrap.dedent(module_code).lstrip(),
    encoding="utf-8",
)

print("\n[1] CREATED SELECTION MODULE")
print(MODULE)

# ------------------------------------------------
# 2. Write independent correctness tests
# ------------------------------------------------

test_code = r'''
import sys
import json
from pathlib import Path

import torch

ROOT = Path("/kaggle/working/B2_FastV")
sys.path.insert(0, str(ROOT))

from fastv_selection import (
    compute_visual_importance,
    select_fastv_indices,
    gather_sequence,
    gather_causal_attention_mask,
    RETENTION_TO_COUNT,
)

torch.manual_seed(2026)

print("=" * 76)
print("B2 — CELL 6: FASTV SELECTION UNIT TESTS")
print("=" * 76)

VISUAL_START = 8
VISUAL_COUNT = 576
SUFFIX_COUNT = 12

LENGTH = VISUAL_START + VISUAL_COUNT + SUFFIX_COUNT

# ------------------------------------------------
# 1. Construct valid causal attention
# ------------------------------------------------

print("\n[1] CONSTRUCT CAUSAL ATTENTION")

random_scores = torch.randn(
    1, 4, LENGTH, LENGTH
)

future_positions = torch.triu(
    torch.ones(
        LENGTH, LENGTH,
        dtype=torch.bool,
    ),
    diagonal=1,
)

random_scores = random_scores.masked_fill(
    future_positions[None, None],
    float("-inf"),
)

attention = torch.softmax(
    random_scores,
    dim=-1,
)

assert torch.isfinite(attention).all()

assert torch.all(
    attention[0, :, future_positions] == 0
)

assert torch.allclose(
    attention.sum(dim=-1),
    torch.ones_like(attention.sum(dim=-1)),
    atol=1e-6,
)

print("Causal attention: PASS")

# ------------------------------------------------
# 2. Visual importance
# ------------------------------------------------

print("\n[2] VISUAL IMPORTANCE")

importance = compute_visual_importance(
    attention,
    VISUAL_START,
)

assert importance.shape == (1, 576)
assert torch.isfinite(importance).all()

print("Importance score shape:", tuple(importance.shape))
print("Attention aggregation: PASS")

# ------------------------------------------------
# 3. All three FastV retention levels
# ------------------------------------------------

print("\n[3] RETENTION AND TEXT PRESERVATION")

expected_lengths = {
    75: 452,
    50: 308,
    25: 164,
}

for pct in (75, 50, 25):

    indices, scores = select_fastv_indices(
        attention,
        VISUAL_START,
        pct,
    )

    expected_visual = RETENTION_TO_COUNT[pct]
    expected_length = expected_lengths[pct]

    assert indices.shape == (1, expected_length)

    kept = indices[0]

    visual = kept[
        (kept >= VISUAL_START)
        & (kept < VISUAL_START + VISUAL_COUNT)
    ]

    assert visual.numel() == expected_visual

    # Prefix text: preserved exactly.
    assert torch.equal(
        kept[:VISUAL_START],
        torch.arange(VISUAL_START),
    )

    # Suffix text: preserved exactly.
    assert torch.equal(
        kept[-SUFFIX_COUNT:],
        torch.arange(
            VISUAL_START + VISUAL_COUNT,
            LENGTH,
        ),
    )

    # No reordering or duplicate positions.
    assert torch.all(kept[1:] > kept[:-1])

    # Check physical hidden-state selection.
    hidden = torch.arange(
        LENGTH,
        dtype=torch.float32
    ).view(1, LENGTH, 1).repeat(1, 1, 4)

    selected_hidden = gather_sequence(
        hidden,
        indices,
    )

    assert selected_hidden.shape == (
        1, expected_length, 4
    )

    assert torch.equal(
        selected_hidden[0, :, 0].long(),
        kept,
    )

    # Check original position IDs are preserved.
    positions = torch.arange(
        LENGTH
    ).unsqueeze(0)

    selected_positions = gather_sequence(
        positions,
        indices,
    )

    assert torch.equal(selected_positions[0], kept)

    print(
        f"{pct}%: visual={expected_visual}, "
        f"sequence={expected_length} — PASS"
    )

# ------------------------------------------------
# 4. Causal mask reduction
# ------------------------------------------------

print("\n[4] REDUCED CAUSAL MASK")

mask = torch.zeros(
    1, 1, LENGTH, LENGTH
)

mask.masked_fill_(
    future_positions[None, None],
    torch.finfo(torch.float32).min,
)

for pct in (75, 50, 25):

    indices, _ = select_fastv_indices(
        attention,
        VISUAL_START,
        pct,
    )

    reduced = gather_causal_attention_mask(
        mask,
        indices,
    )

    reduced_length = expected_lengths[pct]

    assert reduced.shape == (
        1, 1, reduced_length, reduced_length
    )

    lower = torch.tril(
        torch.ones(
            reduced_length,
            reduced_length,
            dtype=torch.bool,
        )
    )

    upper = ~lower

    actual = reduced[0, 0]

    assert torch.all(actual[lower] == 0)
    assert torch.all(actual[upper] < -1e20)

    print(
        f"{pct}%: causal-mask shape "
        f"{tuple(reduced.shape)} — PASS"
    )

# ------------------------------------------------
# 5. Verify that attention controls selection
# ------------------------------------------------

print("\n[5] ATTENTION-SENSITIVE RANKING")

def make_controlled_attention(target_offset):
    """
    Create a valid causal attention distribution.

    The last text queries direct their attention
    to one selected group of 144 visual tokens.
    """
    controlled = torch.zeros(
        1, 2, LENGTH, LENGTH
    )

    # Default: attend to position zero,
    # which is visible to every query.
    controlled[..., 0] = 1.0

    target_start = VISUAL_START + target_offset
    target_end = target_start + 144

    final_text_start = VISUAL_START + VISUAL_COUNT

    controlled[
        :, :,
        final_text_start:,
        0
    ] = 0.0

    controlled[
        :, :,
        final_text_start:,
        target_start:target_end
    ] = 1.0 / 144

    assert torch.allclose(
        controlled.sum(dim=-1),
        torch.ones_like(controlled.sum(dim=-1)),
    )

    assert torch.all(
        controlled[0, :, future_positions] == 0
    )

    return controlled


attention_a = make_controlled_attention(0)
attention_b = make_controlled_attention(144)

indices_a, _ = select_fastv_indices(
    attention_a,
    VISUAL_START,
    25,
)

indices_b, _ = select_fastv_indices(
    attention_b,
    VISUAL_START,
    25,
)

def visual_positions(indices):
    values = indices[0]
    return values[
        (values >= VISUAL_START)
        & (values < VISUAL_START + VISUAL_COUNT)
    ]


visual_a = visual_positions(indices_a)
visual_b = visual_positions(indices_b)

assert torch.equal(
    visual_a,
    torch.arange(
        VISUAL_START,
        VISUAL_START + 144,
    ),
)

assert torch.equal(
    visual_b,
    torch.arange(
        VISUAL_START + 144,
        VISUAL_START + 288,
    ),
)

assert not torch.equal(visual_a, visual_b)

print("First visual region selected: PASS")
print("Second visual region selected: PASS")
print("Attention-dependent ranking: PASS")

# ------------------------------------------------
# 6. Reject unsupported retention levels
# ------------------------------------------------

print("\n[6] PROTOCOL GUARD")

for invalid_pct in (0, 10, 100):
    try:
        select_fastv_indices(
            attention,
            VISUAL_START,
            invalid_pct,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"Unsupported retention accepted: {invalid_pct}"
        )

print("Only 75%, 50%, 25% accepted: PASS")

# ------------------------------------------------
# 7. Save verified report
# ------------------------------------------------

print("\n[7] SAVE UNIT-TEST REPORT")

report = {
    "experiment": "B2_FastV",
    "test": "attention_guided_visual_token_selection",
    "status": "PASS",
    "pruning_after_layer": 2,
    "visual_tokens_original": 576,
    "retained_visual_tokens": {
        "75": 432,
        "50": 288,
        "25": 144,
    },
    "causal_attention_test": True,
    "attention_sensitive_ranking_test": True,
    "all_text_tokens_preserved": True,
    "original_token_order_preserved": True,
    "original_position_ids_preserved": True,
    "physical_hidden_state_gather_test": True,
    "reduced_causal_mask_test": True,
    "seven_billion_model_tested": False,
}

path = ROOT / "configs/fastv_selection_test.json"

path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:", path)

print("\n" + "=" * 76)
print("CELL 6 PASSED — FASTV SELECTION VERIFIED")
print("=" * 76)
'''

TEST.parent.mkdir(parents=True, exist_ok=True)
TEST.write_text(
    textwrap.dedent(test_code).lstrip(),
    encoding="utf-8",
)

print("\n[2] CREATED VERIFICATION SCRIPT")
print(TEST)

# ------------------------------------------------
# 3. Syntax verification
# ------------------------------------------------

print("\n[3] VERIFY PYTHON SYNTAX")

for path in (MODULE, TEST):
    subprocess.run(
        [str(PYTHON), "-m", "py_compile", str(path)],
        check=True,
    )
    print("Syntax PASS:", path.name)

# ------------------------------------------------
# 4. Run independent tests
# ------------------------------------------------

print("\n[4] EXECUTE FASTV SELECTION TESTS\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(ROOT),
    str(REPO),
    env.get("PYTHONPATH", ""),
])

result = subprocess.run(
    [str(PYTHON), "-u", str(TEST)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 6 selection tests failed. "
        "Share the traceback before proceeding."
    )

assert REPORT.is_file()

print("\n" + "=" * 76)
print("B2 — CELL 6 COMPLETE")
print("=" * 76)
print("Module:", MODULE)
print("Unit tests:", TEST)
print("Report:", REPORT)


B2 — CELL 6: FASTV TOKEN SELECTION

[1] CREATED SELECTION MODULE
/kaggle/working/B2_FastV/fastv_selection.py

[2] CREATED VERIFICATION SCRIPT
/kaggle/working/B2_FastV/scripts/cell6_test_fastv_selection.py

[3] VERIFY PYTHON SYNTAX
Syntax PASS: fastv_selection.py
Syntax PASS: cell6_test_fastv_selection.py

[4] EXECUTE FASTV SELECTION TESTS

B2 — CELL 6: FASTV SELECTION UNIT TESTS

[1] CONSTRUCT CAUSAL ATTENTION
Causal attention: PASS

[2] VISUAL IMPORTANCE
Importance score shape: (1, 576)
Attention aggregation: PASS

[3] RETENTION AND TEXT PRESERVATION
75%: visual=432, sequence=452 — PASS
50%: visual=288, sequence=308 — PASS
25%: visual=144, sequence=164 — PASS

[4] REDUCED CAUSAL MASK
75%: causal-mask shape (1, 1, 452, 452) — PASS
50%: causal-mask shape (1, 1, 308, 308) — PASS
25%: causal-mask shape (1, 1, 164, 164) — PASS

[5] ATTENTION-SENSITIVE RANKING
First visual region selected: PASS
Second visual region selected: PASS
Attention-dependent ranking: PASS

[6] PROTOCOL GUARD
Only 75

In [9]:

# ================================================================
# B2 — CELL 7: FASTV PREFILL IMPLEMENTATION AND VALIDATION
# Environment: PyTorch 2.1.2 / Transformers 4.36.2
# ================================================================

from pathlib import Path
import subprocess
import textwrap
import os
import json

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

MODULE = ROOT / "fastv_prefill.py"
TEST = ROOT / "scripts/cell7_test_fastv_prefill.py"
REPORT = ROOT / "configs/fastv_prefill_test.json"

assert PYTHON.is_file()
assert (ROOT / "fastv_selection.py").is_file()
assert (ROOT / "configs/fastv_selection_test.json").is_file()

print("=" * 76)
print("B2 — CELL 7: FASTV PREFILL")
print("=" * 76)

# ------------------------------------------------
# 1. Write prefill implementation
# ------------------------------------------------

module_code = r'''
import torch
from contextlib import contextmanager

from fastv_selection import (
    select_fastv_indices,
    gather_sequence,
    gather_causal_attention_mask,
)


def create_causal_mask(batch, length, dtype, device):
    """
    Additive causal attention mask.

    Shape: [batch, 1, sequence, sequence].
    """
    assert dtype.is_floating_point

    future = torch.ones(
        length, length,
        device=device,
        dtype=torch.bool,
    ).triu(diagonal=1)

    mask = torch.zeros(
        batch, 1, length, length,
        device=device,
        dtype=dtype,
    )

    return mask.masked_fill(
        future[None, None],
        torch.finfo(dtype).min,
    )


@contextmanager
def preserve_original_rope_positions(
    layer, position_ids
):
    """
    Transformers 4.36.2 Mistral computes RoPE lookup
    tables using the current attention sequence length.

    After FastV removes tokens, the remaining original
    positions may exceed that shortened length.

    Temporarily extend the rotary lookup to cover the
    highest ORIGINAL position, without renumbering tokens.
    Restore the method even when an exception occurs.
    """
    rotary = layer.self_attn.rotary_emb
    original_forward = rotary.forward

    max_position = int(position_ids.max().item()) + 1

    def extended_forward(x, seq_len=None):
        requested = (
            int(seq_len)
            if seq_len is not None
            else int(x.shape[-2])
        )
        return original_forward(
            x,
            seq_len=max(requested, max_position),
        )

    rotary.forward = extended_forward

    try:
        yield
    finally:
        rotary.forward = original_forward


@torch.inference_mode()
def fastv_prefill(
    model,
    inputs_embeds,
    visual_start,
    retention_pct,
    visual_count=576,
    pruning_after_layer=2,
    allow_unpruned_control=False,
):
    """
    Attention-guided FastV decoder prefill.

    First K decoder layers: full sequence.
    After layer K: physically remove low-ranked
    visual tokens. All text tokens are preserved.

    The 100% mode is strictly a native-equivalence
    test, not a B2 experimental retention.

    Works with single-GPU or model-parallel layers.
    No KV caching here; Cell 8 implements caching.
    """
    if retention_pct == 100:
        if not allow_unpruned_control:
            raise ValueError(
                "100% is reserved for verification."
            )
    elif retention_pct not in (75, 50, 25):
        raise ValueError(
            "Supported retentions: 75, 50, 25."
        )

    if inputs_embeds.ndim != 3:
        raise ValueError("Expected [batch, seq, hidden].")

    if inputs_embeds.shape[0] != 1:
        raise ValueError(
            "This implementation requires batch size 1."
        )

    if visual_count != 576:
        raise ValueError(
            "Expected 576 projected visual tokens."
        )

    layers = model.model.layers
    n_layers = len(layers)

    assert 0 < pruning_after_layer < n_layers

    batch, full_length, _ = inputs_embeds.shape

    assert (
        0 <= visual_start <= full_length - visual_count
    )

    hidden = inputs_embeds

    position_ids = torch.arange(
        full_length,
        dtype=torch.long,
        device=hidden.device,
    ).unsqueeze(0)

    attention_mask = create_causal_mask(
        batch,
        full_length,
        hidden.dtype,
        hidden.device,
    )

    keep_indices = None
    visual_importance = None
    layer_lengths = []
    layer_position_max = []

    for index, layer in enumerate(layers):
        # Respect the actual GPU placement of every layer.
        device = next(layer.parameters()).device

        hidden = hidden.to(device)
        position_ids = position_ids.to(device)
        attention_mask = attention_mask.to(device)

        layer_lengths.append(int(hidden.shape[1]))
        layer_position_max.append(
            int(position_ids.max().item())
        )

        collect_attention = (
            index == pruning_after_layer - 1
            and retention_pct != 100
        )

        with preserve_original_rope_positions(
            layer, position_ids
        ):
            result = layer(
                hidden,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=None,
                output_attentions=collect_attention,
                use_cache=False,
            )

        hidden = result[0]

        if collect_attention:
            # Attention from the final full-sequence
            # layer determines which visual tokens stay.
            weights = result[1]

            if weights is None:
                raise RuntimeError(
                    "Decoder did not return attention."
                )

            assert weights.shape[-2:] == (
                full_length, full_length
            )

            keep_indices, visual_importance = (
                select_fastv_indices(
                    attention_weights=weights,
                    visual_start=visual_start,
                    retention_pct=retention_pct,
                    visual_count=visual_count,
                )
            )

            # Physical token removal, NOT masking alone.
            hidden = gather_sequence(
                hidden, keep_indices
            )

            # Keep absolute indices from full sequence.
            position_ids = gather_sequence(
                position_ids, keep_indices
            )

            # Retain valid causal relationships.
            attention_mask = (
                gather_causal_attention_mask(
                    attention_mask,
                    keep_indices,
                )
            )

    norm_device = next(
        model.model.norm.parameters()
    ).device

    hidden = model.model.norm(
        hidden.to(norm_device)
    )

    head_device = next(
        model.lm_head.parameters()
    ).device

    logits = model.lm_head(
        hidden.to(head_device)
    ).float()

    retained_visual = (
        visual_count
        if retention_pct == 100
        else visual_count * retention_pct // 100
    )

    expected_short = (
        full_length - visual_count + retained_visual
    )

    expected_lengths = [
        full_length if i < pruning_after_layer
        else expected_short
        for i in range(n_layers)
    ]

    assert layer_lengths == expected_lengths

    assert logits.shape[:2] == (
        batch, expected_short
    )

    # Original final text position is unchanged.
    assert int(position_ids[0, -1]) == full_length - 1

    if retention_pct != 100:
        assert keep_indices is not None
        assert visual_importance is not None

    return {
        "logits": logits,
        "last_logits": logits[:, -1, :],
        "keep_indices": keep_indices,
        "visual_importance": visual_importance,
        "position_ids": position_ids,
        "layer_sequence_lengths": layer_lengths,
        "layer_max_position_ids": layer_position_max,
        "original_sequence_length": full_length,
        "final_sequence_length": expected_short,
        "retained_visual_tokens": retained_visual,
    }
'''

MODULE.write_text(
    textwrap.dedent(module_code).lstrip(),
    encoding="utf-8"
)

print("\n[1] CREATED PREFILL MODULE")
print(MODULE)

# ------------------------------------------------
# 2. Write independent integration tests
# ------------------------------------------------

test_code = r'''
import sys
import json
from pathlib import Path

import torch
from transformers import (
    MistralConfig,
    MistralForCausalLM,
)

ROOT = Path("/kaggle/working/B2_FastV")
sys.path.insert(0, str(ROOT))

from fastv_prefill import (
    fastv_prefill,
    create_causal_mask,
    preserve_original_rope_positions,
)

torch.manual_seed(2026)
torch.set_num_threads(4)

print("=" * 76)
print("B2 — CELL 7: REAL MISTRAL DECODER INTEGRATION")
print("=" * 76)

# ------------------------------------------------
# 1. Small native Mistral model
# ------------------------------------------------

print("\n[1] BUILD FOUR-LAYER MISTRAL")

config = MistralConfig(
    vocab_size=512,
    hidden_size=128,
    intermediate_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    max_position_embeddings=2048,
    sliding_window=None,
    attention_dropout=0.0,
    use_cache=False,
)

model = MistralForCausalLM(config).eval()
model.requires_grad_(False)

PREFIX = 4
VISUAL = 576
SUFFIX = 12
LENGTH = PREFIX + VISUAL + SUFFIX

# Create an artificial multimodal embedding sequence.
# This tests the decoder, not the CLIP image encoder.
with torch.inference_mode():
    prefix_ids = torch.randint(
        0, config.vocab_size, (1, PREFIX)
    )
    suffix_ids = torch.randint(
        0, config.vocab_size, (1, SUFFIX)
    )

    prefix = model.model.embed_tokens(prefix_ids)
    suffix = model.model.embed_tokens(suffix_ids)

    visual = torch.randn(
        1, VISUAL, config.hidden_size
    ) * 0.02

    embeddings = torch.cat([
        prefix, visual, suffix
    ], dim=1)

assert embeddings.shape == (1, LENGTH, 128)
print("Original sequence length:", LENGTH)

# ------------------------------------------------
# 2. Native Mistral equivalence, 100% CONTROL ONLY
# ------------------------------------------------

print("\n[2] VERIFY UNPRUNED NATIVE EQUIVALENCE")

with torch.inference_mode():
    native = model(
        inputs_embeds=embeddings,
        attention_mask=torch.ones(
            1, LENGTH, dtype=torch.long
        ),
        use_cache=False,
        return_dict=True,
    ).logits.float()

    custom = fastv_prefill(
        model,
        embeddings,
        visual_start=PREFIX,
        retention_pct=100,
        allow_unpruned_control=True,
    )

assert custom["logits"].shape == native.shape

max_error = (
    custom["logits"] - native
).abs().max().item()

print("Maximum absolute logit difference:", max_error)

assert torch.allclose(
    custom["logits"],
    native,
    atol=1e-5,
    rtol=1e-5,
), "Unpruned custom forward differs from native Mistral."

assert custom["layer_sequence_lengths"] == [
    LENGTH, LENGTH, LENGTH, LENGTH
]

print("100% native equivalence: PASS")

# ------------------------------------------------
# 3. FastV pruning at each retention level
# ------------------------------------------------

print("\n[3] VERIFY ACTUAL FASTV PREFILL")

expected = {
    75: (432, 448),
    50: (288, 304),
    25: (144, 160),
}

results = {}

for pct, (visual_kept, short) in expected.items():

    output = fastv_prefill(
        model,
        embeddings,
        visual_start=PREFIX,
        retention_pct=pct,
    )

    expected_layers = [
        LENGTH, LENGTH, short, short
    ]

    assert output["layer_sequence_lengths"] == (
        expected_layers
    )

    assert output["logits"].shape == (
        1, short, config.vocab_size
    )

    assert output["retained_visual_tokens"] == (
        visual_kept
    )

    keep = output["keep_indices"][0].cpu()
    positions = output["position_ids"][0].cpu()

    # Position IDs remain their ORIGINAL values.
    assert torch.equal(keep, positions)

    # Suffix text positions cannot be removed.
    assert torch.equal(
        positions[-SUFFIX:],
        torch.arange(LENGTH-SUFFIX, LENGTH)
    )

    assert int(positions[-1]) == LENGTH - 1
    assert torch.isfinite(output["logits"]).all()

    # Demonstrate high absolute positions are
    # still present in the shortened layers.
    assert output["layer_max_position_ids"] == [
        LENGTH-1
    ] * 4

    results[str(pct)] = {
        "visual_tokens": visual_kept,
        "short_length": short,
        "layer_lengths": expected_layers,
        "last_absolute_position": int(positions[-1]),
    }

    print(
        f"{pct}%: {visual_kept} visual tokens, "
        f"layers={expected_layers}, "
        f"last position={int(positions[-1])} — PASS"
    )

# ------------------------------------------------
# 4. Repeated-call safety
# ------------------------------------------------

print("\n[4] VERIFY ROTARY PATCH RESTORATION")

for layer in model.model.layers:
    assert not hasattr(
        layer.self_attn.rotary_emb,
        "_b2_original_forward"
    )

second = fastv_prefill(
    model,
    embeddings,
    visual_start=PREFIX,
    retention_pct=50,
)

assert second["layer_sequence_lengths"] == [
    LENGTH, LENGTH, 304, 304
]

print("Repeated inference: PASS")
print("No persistent rotary modification: PASS")

# ------------------------------------------------
# 5. Save result only after all assertions pass
# ------------------------------------------------

print("\n[5] SAVE PREFILL REPORT")

report = {
    "experiment": "B2_FastV",
    "test": "tiny_mistral_fastv_prefill",
    "status": "PASS",
    "torch_native_max_logit_error": max_error,
    "native_unpruned_equivalence": True,
    "pruning_after_layer": 2,
    "original_sequence_length": LENGTH,
    "original_visual_tokens": 576,
    "results": results,
    "original_rope_positions_preserved": True,
    "all_text_tokens_preserved": True,
    "physical_sequence_reduction_verified": True,
    "heterogeneous_kv_cache_tested": False,
    "full_7b_model_tested": False,
}

path = ROOT / "configs/fastv_prefill_test.json"
path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", path)

print("\n" + "=" * 76)
print("CELL 7 PASSED — FASTV PREFILL VERIFIED")
print("=" * 76)
'''

TEST.parent.mkdir(parents=True, exist_ok=True)
TEST.write_text(
    textwrap.dedent(test_code).lstrip(),
    encoding="utf-8"
)

print("\n[2] CREATED PREFILL TEST")
print(TEST)

# ------------------------------------------------
# 3. Syntax and runtime validation
# ------------------------------------------------

print("\n[3] VERIFY PYTHON SYNTAX")

for file in (MODULE, TEST):
    subprocess.run(
        [str(PYTHON), "-m", "py_compile", str(file)],
        check=True
    )
    print("Syntax PASS:", file.name)

print("\n[4] EXECUTE PREFILL UNIT TESTS\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(ROOT),
    str(REPO),
    env.get("PYTHONPATH", ""),
])

env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(TEST)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 7 prefill validation failed. "
        "Share the traceback before continuing."
    )

assert REPORT.is_file()

report = json.loads(
    REPORT.read_text(encoding="utf-8")
)
assert report["status"] == "PASS"

print("\n" + "=" * 76)
print("B2 — CELL 7 COMPLETE")
print("=" * 76)
print("Prefill module:", MODULE)
print("Validation report:", REPORT)
print("Experimental retentions: 75%, 50%, 25%")


B2 — CELL 7: FASTV PREFILL

[1] CREATED PREFILL MODULE
/kaggle/working/B2_FastV/fastv_prefill.py

[2] CREATED PREFILL TEST
/kaggle/working/B2_FastV/scripts/cell7_test_fastv_prefill.py

[3] VERIFY PYTHON SYNTAX
Syntax PASS: fastv_prefill.py
Syntax PASS: cell7_test_fastv_prefill.py

[4] EXECUTE PREFILL UNIT TESTS

B2 — CELL 7: REAL MISTRAL DECODER INTEGRATION

[1] BUILD FOUR-LAYER MISTRAL
Original sequence length: 592

[2] VERIFY UNPRUNED NATIVE EQUIVALENCE
Maximum absolute logit difference: 0.0
100% native equivalence: PASS

[3] VERIFY ACTUAL FASTV PREFILL
75%: 432 visual tokens, layers=[592, 592, 448, 448], last position=591 — PASS
50%: 288 visual tokens, layers=[592, 592, 304, 304], last position=591 — PASS
25%: 144 visual tokens, layers=[592, 592, 160, 160], last position=591 — PASS

[4] VERIFY ROTARY PATCH RESTORATION
Repeated inference: PASS
No persistent rotary modification: PASS

[5] SAVE PREFILL REPORT
Saved: /kaggle/working/B2_FastV/configs/fastv_prefill_test.json

CELL 7 PASSE

In [10]:

# ================================================================
# B2 — CELL 8: CACHE-AWARE FASTV DECODING
#
# Creates:
#   fastv_cached.py
#   scripts/cell8_test_fastv_cached.py
#   configs/fastv_cached_test.json
#
# Uses a small 4-layer Mistral model.
# Does not load the 7B checkpoint.
# ================================================================

from pathlib import Path
import subprocess
import textwrap
import os
import json

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

MODULE = ROOT / "fastv_cached.py"
TEST = ROOT / "scripts/cell8_test_fastv_cached.py"
REPORT = ROOT / "configs/fastv_cached_test.json"

assert PYTHON.is_file()
assert (ROOT / "fastv_selection.py").is_file()
assert (ROOT / "fastv_prefill.py").is_file()
assert (ROOT / "configs/fastv_prefill_test.json").is_file()

print("=" * 76)
print("B2 — CELL 8: HETEROGENEOUS KV-CACHE")
print("=" * 76)

# ------------------------------------------------
# 1. Write cache-aware FastV implementation
# ------------------------------------------------

module_code = r'''
import torch
from transformers.cache_utils import DynamicCache

from fastv_selection import (
    select_fastv_indices,
    gather_sequence,
    gather_causal_attention_mask,
)

from fastv_prefill import (
    create_causal_mask,
    preserve_original_rope_positions,
)


@torch.inference_mode()
def fastv_cached_prefill(
    model,
    inputs_embeds,
    visual_start,
    retention_pct,
    visual_count=576,
    pruning_after_layer=2,
    allow_unpruned_control=False,
):
    """
    Cache-aware FastV prefill.

    Layers 0 through K-1:
        Process and cache every original token.

    Layers K through N-1:
        Process and cache only retained tokens.

    No visual KV entries are removed retroactively
    from the first K layers.

    Supports single-example model-parallel inference.
    """

    if retention_pct == 100:
        if not allow_unpruned_control:
            raise ValueError(
                "100% retention is verification-only."
            )
    elif retention_pct not in (75, 50, 25):
        raise ValueError(
            "Supported retentions: 75, 50, 25."
        )

    if inputs_embeds.ndim != 3:
        raise ValueError(
            "Expected inputs_embeds [B,S,H]."
        )

    if inputs_embeds.shape[0] != 1:
        raise ValueError("Batch size must be 1.")

    if visual_count != 576:
        raise ValueError(
            "Expected exactly 576 visual tokens."
        )

    layers = model.model.layers
    layer_count = len(layers)

    if not 0 < pruning_after_layer < layer_count:
        raise ValueError("Invalid pruning layer.")

    batch, original_length, _ = (
        inputs_embeds.shape
    )

    if not (
        0 <= visual_start
        <= original_length - visual_count
    ):
        raise ValueError("Invalid visual span.")

    hidden = inputs_embeds

    positions = torch.arange(
        original_length,
        dtype=torch.long,
        device=hidden.device,
    ).unsqueeze(0)

    attention_mask = create_causal_mask(
        batch,
        original_length,
        hidden.dtype,
        hidden.device,
    )

    cache = DynamicCache()

    layer_lengths = []
    keep_indices = None
    visual_importance = None

    for index, layer in enumerate(layers):

        device = next(layer.parameters()).device

        hidden = hidden.to(device)
        positions = positions.to(device)
        attention_mask = attention_mask.to(device)

        current_length = hidden.shape[1]

        layer_lengths.append(
            int(current_length)
        )

        collect_attention = (
            index == pruning_after_layer - 1
            and retention_pct != 100
        )

        # Preserve original absolute RoPE positions.
        with preserve_original_rope_positions(
            layer, positions
        ):
            output = layer(
                hidden,
                attention_mask=attention_mask,
                position_ids=positions,
                past_key_value=cache,
                output_attentions=collect_attention,
                use_cache=True,
            )

        hidden = output[0]

        # The layer has now cached its own input
        # sequence, before FastV removes anything.
        actual_cache_length = (
            cache.get_seq_length(index)
        )

        if actual_cache_length != current_length:
            raise AssertionError(
                f"Layer {index} cache length: "
                f"expected {current_length}, "
                f"found {actual_cache_length}"
            )

        if collect_attention:

            attention_weights = output[1]

            if attention_weights is None:
                raise RuntimeError(
                    "Layer did not return attention."
                )

            if attention_weights.shape[-2:] != (
                original_length,
                original_length,
            ):
                raise AssertionError(
                    "Unexpected ranking-attention shape."
                )

            keep_indices, visual_importance = (
                select_fastv_indices(
                    attention_weights,
                    visual_start,
                    retention_pct,
                    visual_count,
                )
            )

            # Physically reduce inputs to layer K.
            hidden = gather_sequence(
                hidden, keep_indices
            )

            # Do not renumber original token positions.
            positions = gather_sequence(
                positions, keep_indices
            )

            # Physically reduce the attention mask.
            attention_mask = (
                gather_causal_attention_mask(
                    attention_mask,
                    keep_indices,
                )
            )

    retained_visual = (
        visual_count
        if retention_pct == 100
        else visual_count * retention_pct // 100
    )

    reduced_length = (
        original_length
        - visual_count
        + retained_visual
    )

    expected_lengths = [
        (
            original_length
            if i < pruning_after_layer
            else reduced_length
        )
        for i in range(layer_count)
    ]

    assert layer_lengths == expected_lengths

    cache_lengths = [
        int(cache.get_seq_length(i))
        for i in range(layer_count)
    ]

    assert cache_lengths == expected_lengths

    assert int(positions[0, -1]) == (
        original_length - 1
    )

    # Model-parallel norm and language-model head.
    norm_device = next(
        model.model.norm.parameters()
    ).device

    hidden = model.model.norm(
        hidden.to(norm_device)
    )

    head_device = next(
        model.lm_head.parameters()
    ).device

    last_hidden = hidden[:, -1:, :].to(
        head_device
    )

    last_logits = model.lm_head(
        last_hidden
    ).float()[:, -1, :]

    assert torch.isfinite(last_logits).all()

    return {
        "last_logits": last_logits,
        "cache": cache,
        "original_length": original_length,
        "generated_count": 0,
        "pruning_after_layer": pruning_after_layer,
        "retention_pct": retention_pct,
        "retained_visual_tokens": retained_visual,
        "reduced_length": reduced_length,
        "layer_sequence_lengths": layer_lengths,
        "prefill_cache_lengths": cache_lengths,
        "keep_indices": keep_indices,
        "visual_importance": visual_importance,
    }


@torch.inference_mode()
def fastv_cached_decode_step(
    model,
    next_token_ids,
    state,
):
    """
    Process one new text token using the existing
    heterogeneous KV cache.

    The new token's position is measured relative
    to the ORIGINAL unpruned prompt length.

    Each layer sees its own cached KV history.
    """

    if next_token_ids.shape != (1, 1):
        raise ValueError(
            "Expected one token with shape [1,1]."
        )

    cache = state["cache"]

    original_length = state["original_length"]
    generated_count = state["generated_count"]

    absolute_position = (
        original_length + generated_count
    )

    embedding_device = next(
        model.model.embed_tokens.parameters()
    ).device

    token = next_token_ids.to(
        embedding_device,
        dtype=torch.long,
    )

    hidden = model.model.embed_tokens(token)

    position_ids = torch.tensor(
        [[absolute_position]],
        device=embedding_device,
        dtype=torch.long,
    )

    layers = model.model.layers

    previous_lengths = [
        int(cache.get_seq_length(i))
        for i in range(len(layers))
    ]

    for index, layer in enumerate(layers):

        device = next(layer.parameters()).device

        hidden = hidden.to(device)
        local_positions = position_ids.to(device)

        # One new token may attend to every token
        # in its layer-specific cache. There are
        # no future keys in a single-token step.
        with preserve_original_rope_positions(
            layer, local_positions
        ):
            output = layer(
                hidden,
                attention_mask=None,
                position_ids=local_positions,
                past_key_value=cache,
                output_attentions=False,
                use_cache=True,
            )

        hidden = output[0]

        if hidden.shape[1] != 1:
            raise AssertionError(
                "Decode must process one token."
            )

        new_length = int(
            cache.get_seq_length(index)
        )

        if new_length != previous_lengths[index] + 1:
            raise AssertionError(
                f"KV cache failed to grow at "
                f"layer {index}: "
                f"{previous_lengths[index]} -> "
                f"{new_length}"
            )

    norm_device = next(
        model.model.norm.parameters()
    ).device

    hidden = model.model.norm(
        hidden.to(norm_device)
    )

    head_device = next(
        model.lm_head.parameters()
    ).device

    logits = model.lm_head(
        hidden.to(head_device)
    ).float()[:, -1, :]

    assert torch.isfinite(logits).all()

    # Inference state is deliberately mutable:
    # the cache object already contains the
    # newly processed token.
    state["generated_count"] += 1

    return logits, state
'''

MODULE.write_text(
    textwrap.dedent(module_code).lstrip(),
    encoding="utf-8",
)

print("\n[1] CREATED CACHED DECODER")
print(MODULE)

# ------------------------------------------------
# 2. Write independent correctness tests
# ------------------------------------------------

test_code = r'''
import sys
import json
from pathlib import Path

import torch
from transformers import (
    MistralConfig,
    MistralForCausalLM,
)
from transformers.cache_utils import DynamicCache

ROOT = Path("/kaggle/working/B2_FastV")
sys.path.insert(0, str(ROOT))

from fastv_cached import (
    fastv_cached_prefill,
    fastv_cached_decode_step,
)

torch.manual_seed(2026)
torch.set_num_threads(4)

print("=" * 76)
print("B2 — CELL 8: CACHED FASTV DECODING TEST")
print("=" * 76)

# ------------------------------------------------
# 1. Build four-layer Mistral
# ------------------------------------------------

print("\n[1] BUILD SMALL MISTRAL MODEL")

config = MistralConfig(
    vocab_size=512,
    hidden_size=128,
    intermediate_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    max_position_embeddings=2048,
    sliding_window=None,
    attention_dropout=0.0,
    use_cache=True,
)

model = MistralForCausalLM(config).eval()
model.requires_grad_(False)

PREFIX = 4
VISUAL = 576
SUFFIX = 12
FULL = PREFIX + VISUAL + SUFFIX

with torch.inference_mode():

    prefix = model.model.embed_tokens(
        torch.randint(0, 512, (1, PREFIX))
    )

    visual = (
        torch.randn(1, VISUAL, 128) * 0.02
    )

    suffix = model.model.embed_tokens(
        torch.randint(0, 512, (1, SUFFIX))
    )

    embeddings = torch.cat(
        [prefix, visual, suffix],
        dim=1,
    )

assert embeddings.shape == (1, 592, 128)

print("Input tokens:", FULL)
print("Layers:", len(model.model.layers))

# ------------------------------------------------
# 2. Native cached prefill, 100% control
# ------------------------------------------------

print("\n[2] NATIVE CACHED PREFILL IDENTITY")

native_cache = DynamicCache()

with torch.inference_mode():
    native_output = model(
        inputs_embeds=embeddings,
        attention_mask=torch.ones(
            1, FULL, dtype=torch.long
        ),
        past_key_values=native_cache,
        use_cache=True,
        return_dict=True,
    )

native_logits = native_output.logits[
    :, -1, :
].float()

custom = fastv_cached_prefill(
    model,
    embeddings,
    visual_start=PREFIX,
    retention_pct=100,
    allow_unpruned_control=True,
)

custom_logits = custom["last_logits"]

max_prefill_error = (
    native_logits - custom_logits
).abs().max().item()

print(
    "Maximum prefill logit difference:",
    max_prefill_error,
)

assert torch.allclose(
    native_logits,
    custom_logits,
    atol=1e-5,
    rtol=1e-5,
)

assert custom["prefill_cache_lengths"] == [
    FULL, FULL, FULL, FULL
]

print("Native prefill identity: PASS")

# ------------------------------------------------
# 3. Native autoregressive decoding identity
# ------------------------------------------------

print("\n[3] NATIVE DECODE IDENTITY")

max_decode_error = 0.0
generated_tokens = []

for step in range(8):

    # Greedy token from native model.
    next_id = int(
        native_logits.argmax(dim=-1).item()
    )

    custom_id = int(
        custom_logits.argmax(dim=-1).item()
    )

    assert next_id == custom_id, (
        f"Token mismatch at step {step}"
    )

    generated_tokens.append(next_id)

    # Advance both paths using the SAME token.
    token = torch.tensor(
        [[next_id]],
        dtype=torch.long,
    )

    with torch.inference_mode():
        native_result = model(
            input_ids=token,
            attention_mask=torch.ones(
                1,
                FULL + step + 1,
                dtype=torch.long,
            ),
            position_ids=torch.tensor(
                [[FULL + step]],
                dtype=torch.long,
            ),
            past_key_values=native_cache,
            use_cache=True,
            return_dict=True,
        )

    native_logits = native_result.logits[
        :, -1, :
    ].float()

    custom_logits, custom = (
        fastv_cached_decode_step(
            model,
            token,
            custom,
        )
    )

    error = (
        native_logits - custom_logits
    ).abs().max().item()

    max_decode_error = max(
        max_decode_error, error
    )

    assert torch.allclose(
        native_logits,
        custom_logits,
        atol=1e-5,
        rtol=1e-5,
    ), (
        f"Native/custom decode mismatch "
        f"at step {step + 1}: {error}"
    )

    cache_lengths = [
        custom["cache"].get_seq_length(i)
        for i in range(4)
    ]

    assert cache_lengths == [
        FULL + step + 1
    ] * 4

    print(
        f"Decode step {step+1}: "
        f"token={next_id}, "
        f"max_error={error:.8f} — PASS"
    )

assert custom["generated_count"] == 8

print("Eight-step native equivalence: PASS")

# ------------------------------------------------
# 4. FastV heterogeneous prefill caches
# ------------------------------------------------

print("\n[4] FASTV HETEROGENEOUS CACHES")

expected = {
    75: (432, 448),
    50: (288, 304),
    25: (144, 160),
}

results = {}

for pct, (visual_kept, short) in expected.items():

    state = fastv_cached_prefill(
        model,
        embeddings,
        visual_start=PREFIX,
        retention_pct=pct,
    )

    expected_lengths = [
        FULL, FULL, short, short
    ]

    assert (
        state["prefill_cache_lengths"]
        == expected_lengths
    )

    assert (
        state["layer_sequence_lengths"]
        == expected_lengths
    )

    assert (
        state["retained_visual_tokens"]
        == visual_kept
    )

    print(
        f"{pct}% prefill cache: "
        f"{expected_lengths} — PASS"
    )

    # --------------------------------------------------------
    # 5. Verify three autoregressive decode steps
    # --------------------------------------------------------

    for step, token_id in enumerate(
        (42, 17, 83),
        start=1,
    ):

        token = torch.tensor(
            [[token_id]],
            dtype=torch.long,
        )

        logits, state = fastv_cached_decode_step(
            model,
            token,
            state,
        )

        actual_lengths = [
            state["cache"].get_seq_length(i)
            for i in range(4)
        ]

        expected_step_lengths = [
            FULL + step,
            FULL + step,
            short + step,
            short + step,
        ]

        assert actual_lengths == (
            expected_step_lengths
        )

        assert torch.isfinite(logits).all()

    assert state["generated_count"] == 3

    print(
        f"{pct}% after 3 decode steps: "
        f"{actual_lengths} — PASS"
    )

    results[str(pct)] = {
        "retained_visual_tokens": visual_kept,
        "prefill_cache_lengths": expected_lengths,
        "decode_steps": 3,
        "final_cache_lengths": actual_lengths,
    }

# ------------------------------------------------
# 6. Validate protocol guards
# ------------------------------------------------

print("\n[6] VERIFY RETENTION PROTOCOL")

try:
    fastv_cached_prefill(
        model,
        embeddings,
        visual_start=PREFIX,
        retention_pct=100,
    )
except ValueError:
    print(
        "100% excluded from experimental B2: PASS"
    )
else:
    raise AssertionError(
        "100% experimental run incorrectly permitted."
    )

# ------------------------------------------------
# 7. Save validation report
# ------------------------------------------------

print("\n[7] SAVE CACHED DECODING REPORT")

report = {
    "experiment": "B2_FastV",
    "test": "tiny_mistral_cached_decode",
    "status": "PASS",
    "pruning_after_layer": 2,
    "original_sequence_length": FULL,
    "original_visual_tokens": 576,
    "native_prefill_max_logit_error": (
        max_prefill_error
    ),
    "native_decode_max_logit_error": (
        max_decode_error
    ),
    "native_equivalence_steps": 8,
    "native_equivalence": True,
    "heterogeneous_cache_lengths_verified": True,
    "original_rope_positions_preserved": True,
    "decoder_cache_growth_verified": True,
    "retentions_tested": [75, 50, 25],
    "results": results,
    "full_7b_model_tested": False,
}

path = ROOT / "configs/fastv_cached_test.json"

path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:", path)

print("\n" + "=" * 76)
print(
    "CELL 8 PASSED — CACHED FASTV DECODING VERIFIED"
)
print("=" * 76)
'''

TEST.parent.mkdir(parents=True, exist_ok=True)

TEST.write_text(
    textwrap.dedent(test_code).lstrip(),
    encoding="utf-8",
)

print("\n[2] CREATED CACHED DECODING TEST")
print(TEST)

# ------------------------------------------------
# 3. Syntax validation
# ------------------------------------------------

print("\n[3] VERIFY PYTHON SYNTAX")

for path in (MODULE, TEST):
    subprocess.run(
        [str(PYTHON), "-m", "py_compile", str(path)],
        check=True,
    )

    print("Syntax PASS:", path.name)

# ------------------------------------------------
# 4. Execute tests
# ------------------------------------------------

print("\n[4] EXECUTE CACHE TESTS\n", flush=True)

env = os.environ.copy()

env["PYTHONPATH"] = os.pathsep.join([
    str(ROOT),
    str(REPO),
    env.get("PYTHONPATH", ""),
])

env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(TEST)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 8 cached decoder verification failed. "
        "Share the traceback before continuing."
    )

assert REPORT.is_file()

report = json.loads(
    REPORT.read_text(encoding="utf-8")
)

assert report["status"] == "PASS"

print("\n" + "=" * 76)
print("B2 — CELL 8 COMPLETE")
print("=" * 76)
print("Cached decoder:", MODULE)
print("Validation report:", REPORT)
print("Retentions: 75%, 50%, 25%")


B2 — CELL 8: HETEROGENEOUS KV-CACHE

[1] CREATED CACHED DECODER
/kaggle/working/B2_FastV/fastv_cached.py

[2] CREATED CACHED DECODING TEST
/kaggle/working/B2_FastV/scripts/cell8_test_fastv_cached.py

[3] VERIFY PYTHON SYNTAX
Syntax PASS: fastv_cached.py
Syntax PASS: cell8_test_fastv_cached.py

[4] EXECUTE CACHE TESTS

B2 — CELL 8: CACHED FASTV DECODING TEST

[1] BUILD SMALL MISTRAL MODEL
Input tokens: 592
Layers: 4

[2] NATIVE CACHED PREFILL IDENTITY
Maximum prefill logit difference: 2.384185791015625e-07
Native prefill identity: PASS

[3] NATIVE DECODE IDENTITY
Decode step 1: token=302, max_error=0.00000000 — PASS
Decode step 2: token=336, max_error=0.00000000 — PASS
Decode step 3: token=475, max_error=0.00000000 — PASS
Decode step 4: token=358, max_error=0.00000000 — PASS
Decode step 5: token=115, max_error=0.00000000 — PASS
Decode step 6: token=115, max_error=0.00000000 — PASS
Decode step 7: token=115, max_error=0.00000000 — PASS
Decode step 8: token=115, max_error=0.00000000 — PASS

In [11]:

# ================================================================
# B2 — CELL 9: REAL 7B FASTV SMOKE TEST
#
# 100%: native-equivalence control only
# 75%, 50%, 25%: experimental FastV configurations
#
# No training, no checkpoint modification.
# ================================================================

from pathlib import Path
import subprocess
import textwrap
import json
import os

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"
SCRIPT = ROOT / "scripts/cell9_real_model_smoke.py"
REPORT = ROOT / "configs/real_7b_fastv_smoke_test.json"

for path in [
    PYTHON,
    ROOT / "load_b2_model.py",
    ROOT / "fastv_selection.py",
    ROOT / "fastv_prefill.py",
    ROOT / "fastv_cached.py",
    ROOT / "configs/model_load_report.json",
    ROOT / "configs/fastv_cached_test.json",
]:
    assert path.is_file(), f"Missing prerequisite: {path}"

print("=" * 76)
print("B2 — CELL 9: REAL 7B FASTV VERIFICATION")
print("=" * 76)

script_code = r'''
import sys
import json
import gc
from pathlib import Path

import torch
from PIL import Image
from transformers.cache_utils import DynamicCache

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(REPO))

from load_b2_model import load_b2_model

from fastv_cached import (
    fastv_cached_prefill,
    fastv_cached_decode_step,
)

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
)

from llava.conversation import conv_templates

from llava.mm_utils import (
    tokenizer_image_token,
    process_images,
)

torch.set_grad_enabled(False)

RETENTIONS = (75, 50, 25)

print("=" * 76, flush=True)
print("B2 — REAL LLAVA-MED FASTV SMOKE TEST", flush=True)
print("=" * 76, flush=True)

# ------------------------------------------------------------
# 1. Read verified experiment and dataset
# ------------------------------------------------------------

print("\n[1] VERIFY SLAKE PROTOCOL", flush=True)

protocol = json.loads(
    (ROOT / "configs/experiment_protocol.json")
    .read_text(encoding="utf-8")
)

assert protocol["retention_percentages"] == [
    75, 50, 25
]

dataset_root = Path(protocol["test_json"]).parent
test_json = Path(protocol["test_json"])
image_root = Path(protocol["images_root"])

with test_json.open(encoding="utf-8") as f:
    records = json.load(f)

english = [
    row for row in records
    if str(row.get("q_lang", "")).lower() == "en"
]

assert len(english) == 1061

# Same reference question as our earlier B2 checks.
matches = [
    r for r in english
    if str(r["qid"]) == "11934"
]

assert len(matches) == 1

sample = matches[0]

image_path = image_root / sample["img_name"]
assert image_path.is_file()

print("QID:", sample["qid"], flush=True)
print("Question:", sample["question"], flush=True)
print("Reference:", sample["answer"], flush=True)
print("Image:", image_path, flush=True)

# ------------------------------------------------------------
# 2. Load frozen LLaVA-Med once
# ------------------------------------------------------------

print("\n[2] LOAD FROZEN 7B MODEL", flush=True)

tokenizer, model, vision_tower = load_b2_model()

assert not model.training
assert all(
    not p.requires_grad
    for p in model.parameters()
)

assert torch.cuda.device_count() >= 2

# ------------------------------------------------------------
# 3. Recreate the original multimodal prompt
# ------------------------------------------------------------

print("\n[3] PREPARE MULTIMODAL INPUT", flush=True)

conversation = conv_templates["mistral_instruct"].copy()

conversation.append_message(
    conversation.roles[0],
    DEFAULT_IMAGE_TOKEN + "\n" + sample["question"],
)
conversation.append_message(
    conversation.roles[1],
    None,
)

prompt = conversation.get_prompt()

input_ids = tokenizer_image_token(
    prompt,
    tokenizer,
    IMAGE_TOKEN_INDEX,
    return_tensors="pt",
).unsqueeze(0).to("cuda:0")

image_token_positions = (
    (input_ids[0] == IMAGE_TOKEN_INDEX)
    .nonzero(as_tuple=True)[0]
)

assert image_token_positions.numel() == 1

visual_start = int(
    image_token_positions[0].item()
)

with Image.open(image_path) as file:
    image = file.convert("RGB")

pixels = process_images(
    [image],
    vision_tower.image_processor,
    model.config,
)

if isinstance(pixels, list):
    assert len(pixels) == 1
    pixels = pixels[0].unsqueeze(0)

pixels = pixels.to(
    device="cuda:0",
    dtype=torch.float16,
)

with torch.inference_mode():
    (
        prepared_input_ids,
        prepared_position_ids,
        prepared_attention_mask,
        prepared_past_key_values,
        embeddings,
        prepared_labels,
    ) = model.prepare_inputs_labels_for_multimodal(
        input_ids=input_ids,
        position_ids=None,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        images=pixels,
    )

assert embeddings is not None
assert embeddings.shape[0] == 1
assert embeddings.shape[-1] == 4096

full_length = int(embeddings.shape[1])

visual_count = (
    full_length - input_ids.shape[1] + 1
)

assert visual_count == 576
assert visual_start + 576 <= full_length

print("Image tensor:", tuple(pixels.shape), flush=True)
print("Input token positions:", input_ids.shape[1], flush=True)
print("Multimodal sequence length:", full_length, flush=True)
print("Visual start:", visual_start, flush=True)
print("Visual tokens:", visual_count, flush=True)

# ------------------------------------------------------------
# 4. Native versus custom unpruned identity
# ------------------------------------------------------------

print("\n[4] UNPRUNED NATIVE EQUIVALENCE", flush=True)

torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

native_cache = DynamicCache()

with torch.inference_mode():
    native_result = model(
        inputs_embeds=embeddings,
        attention_mask=torch.ones(
            (1, full_length),
            dtype=torch.long,
            device=embeddings.device,
        ),
        past_key_values=native_cache,
        use_cache=True,
        return_dict=True,
    )

native_last = (
    native_result.logits[:, -1, :].float()
)

custom_state = fastv_cached_prefill(
    model=model,
    inputs_embeds=embeddings,
    visual_start=visual_start,
    retention_pct=100,
    visual_count=576,
    pruning_after_layer=2,
    allow_unpruned_control=True,
)

custom_last = custom_state["last_logits"]

# Two-GPU sharding may put returned logits on
# different devices. Move them ONLY for comparison.
aligned_custom = custom_last.to(
    native_last.device
)

prefill_error = (
    native_last - aligned_custom
).abs().max().item()

assert torch.allclose(
    native_last,
    aligned_custom,
    atol=2e-2,
    rtol=2e-3,
), (
    "Native/custom 100% logits disagree. "
    f"Maximum error={prefill_error}"
)

assert (
    native_last.argmax(dim=-1).item()
    == custom_last.argmax(dim=-1).item()
)

assert custom_state["prefill_cache_lengths"] == [
    full_length
] * 32

print(
    "Maximum absolute logit error:",
    prefill_error,
    flush=True
)
print("Greedy next token: MATCH", flush=True)
print("Native/custom unpruned prefill: PASS", flush=True)

# ------------------------------------------------------------
# 5. Verify one unpruned autoregressive decode step
# ------------------------------------------------------------

print("\n[5] UNPRUNED CACHED DECODE EQUIVALENCE", flush=True)

next_token_id = int(
    native_last.argmax(dim=-1).item()
)

next_token = torch.tensor(
    [[next_token_id]],
    dtype=torch.long,
    device="cuda:0",
)

with torch.inference_mode():
    native_decode = model(
        input_ids=next_token,
        attention_mask=torch.ones(
            (1, full_length + 1),
            dtype=torch.long,
            device="cuda:0",
        ),
        position_ids=torch.tensor(
            [[full_length]],
            dtype=torch.long,
            device="cuda:0",
        ),
        past_key_values=native_cache,
        use_cache=True,
        return_dict=True,
    )

native_decode_logits = (
    native_decode.logits[:, -1, :].float()
)

custom_decode_logits, custom_state = (
    fastv_cached_decode_step(
        model,
        next_token,
        custom_state,
    )
)

aligned_decode = custom_decode_logits.to(
    native_decode_logits.device
)

decode_error = (
    native_decode_logits - aligned_decode
).abs().max().item()

assert torch.allclose(
    native_decode_logits,
    aligned_decode,
    atol=2e-2,
    rtol=2e-3,
), (
    f"Native/custom decode mismatch: {decode_error}"
)

assert (
    native_decode_logits.argmax(dim=-1).item()
    == custom_decode_logits.argmax(dim=-1).item()
)

assert [
    custom_state["cache"].get_seq_length(i)
    for i in range(32)
] == [full_length + 1] * 32

print(
    "Maximum decode logit error:",
    decode_error,
    flush=True
)
print("Unpruned cached decode: PASS", flush=True)

del native_result
del native_decode
del native_cache
del custom_state
del native_last, custom_last
del native_decode_logits, custom_decode_logits
del aligned_custom, aligned_decode

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6. Test actual FastV on the medical image
# ------------------------------------------------------------

print("\n[6] THREE FASTV RETENTIONS", flush=True)

retention_results = {}

for pct in RETENTIONS:

    print(
        f"\n--- {pct}% VISUAL RETENTION ---",
        flush=True,
    )

    state = fastv_cached_prefill(
        model=model,
        inputs_embeds=embeddings,
        visual_start=visual_start,
        retention_pct=pct,
        visual_count=576,
        pruning_after_layer=2,
    )

    kept_visual = 576 * pct // 100

    short_length = (
        full_length - 576 + kept_visual
    )

    expected_cache_lengths = (
        [full_length] * 2
        + [short_length] * 30
    )

    assert state["prefill_cache_lengths"] == (
        expected_cache_lengths
    )

    assert state["retained_visual_tokens"] == (
        kept_visual
    )

    first_token = int(
        state["last_logits"].argmax(dim=-1).item()
    )

    token = torch.tensor(
        [[first_token]],
        dtype=torch.long,
        device="cuda:0",
    )

    next_logits, state = fastv_cached_decode_step(
        model,
        token,
        state,
    )

    expected_after_decode = (
        [full_length + 1] * 2
        + [short_length + 1] * 30
    )

    actual_after_decode = [
        int(state["cache"].get_seq_length(i))
        for i in range(32)
    ]

    assert actual_after_decode == (
        expected_after_decode
    )

    assert torch.isfinite(next_logits).all()

    token_text = tokenizer.decode(
        [first_token],
        skip_special_tokens=True,
    )

    retention_results[str(pct)] = {
        "retention_pct": pct,
        "retained_visual_tokens": kept_visual,
        "removed_visual_tokens": 576 - kept_visual,
        "original_sequence_length": full_length,
        "reduced_sequence_length": short_length,
        "prefill_cache_lengths": expected_cache_lengths,
        "cache_lengths_after_one_decode": (
            actual_after_decode
        ),
        "first_generated_token_id": first_token,
        "first_generated_token_text": token_text,
        "status": "PASS",
    }

    print("Retained visual tokens:", kept_visual, flush=True)
    print("Removed visual tokens:", 576-kept_visual, flush=True)
    print(
        "First 2 layers:",
        full_length,
        "tokens",
        flush=True,
    )
    print(
        "Remaining 30 layers:",
        short_length,
        "tokens",
        flush=True,
    )
    print(
        "First generated token:",
        repr(token_text),
        flush=True,
    )
    print("One-step cache growth: PASS", flush=True)

    del state, next_logits
    gc.collect()
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7. Save verification report
# ------------------------------------------------------------

print("\n[7] SAVE FULL-MODEL VERIFICATION", flush=True)

report = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "model": "microsoft/llava-med-v1.5-mistral-7b",
    "dataset": "SLAKE",
    "qid": str(sample["qid"]),
    "question": sample["question"],
    "reference": sample["answer"],
    "original_visual_tokens": 576,
    "original_sequence_length": full_length,
    "pruning_after_layer": 2,
    "native_unpruned_prefill_max_error": prefill_error,
    "native_unpruned_decode_max_error": decode_error,
    "native_unpruned_equivalence": True,
    "experimental_retentions": [75, 50, 25],
    "retention_results": retention_results,
    "full_answer_generation_tested": False,
    "training_performed": False,
}

path = (
    ROOT / "configs/real_7b_fastv_smoke_test.json"
)

path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:", path, flush=True)

print("\n" + "=" * 76, flush=True)
print("CELL 9 PASSED — REAL FASTV MODEL VERIFIED", flush=True)
print("=" * 76, flush=True)
'''

SCRIPT.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_text(
    textwrap.dedent(script_code).lstrip(),
    encoding="utf-8",
)

print("\n[1] VERIFY SCRIPT SYNTAX", flush=True)

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("Syntax: PASS", flush=True)

print("\n[2] EXECUTE REAL-MODEL TEST\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])
env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(SCRIPT)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 9 real-model verification failed. "
        "Share the traceback before proceeding."
    )

assert REPORT.is_file()

report = json.loads(
    REPORT.read_text(encoding="utf-8")
)

assert report["status"] == "PASS"
assert report["native_unpruned_equivalence"] is True
assert sorted(map(int, report["retention_results"])) == [
    25, 50, 75
]

print("\n" + "=" * 76)
print("B2 — CELL 9 COMPLETE")
print("=" * 76)
print("Frozen 7B model: verified")
print("Unpruned native equivalence: verified")
print("FastV 75%, 50%, 25%: verified")
print("Report:", REPORT)


B2 — CELL 9: REAL 7B FASTV VERIFICATION

[1] VERIFY SCRIPT SYNTAX
Syntax: PASS

[2] EXECUTE REAL-MODEL TEST

B2 — REAL LLAVA-MED FASTV SMOKE TEST

[1] VERIFY SLAKE PROTOCOL
QID: 11934
Question: What modality is used to take this image?
Reference: CT
Image: /kaggle/input/datasets/aqilababy/slaek-data/Slake1.0/imgs/xmlab102/source.jpg

[2] LOAD FROZEN 7B MODEL
B2 — FROZEN TWO-GPU LLAVA-MED LOAD

[1] TOKENIZER
Vocabulary: 32000

[2] LOAD ORIGINAL CHECKPOINT


Loading checkpoint shards: 100%|██████████| 4/4 [00:34<00:00,  8.62s/it]


missing_keys: 0
unexpected_keys: 0
mismatched_keys: 0
error_msgs: 0

[3] VERIFY GPU PLACEMENT
model.embed_tokens: cuda:0 — PASS
model.vision_tower: cuda:0 — PASS
model.mm_projector: cuda:0 — PASS
model.layers.0: cuda:0 — PASS
model.layers.15: cuda:0 — PASS
model.layers.16: cuda:1 — PASS
model.layers.31: cuda:1 — PASS
model.norm: cuda:1 — PASS
lm_head: cuda:1 — PASS

[4] VERIFY VISION WEIGHTS
Vision weight max absolute error: 0.0
Original vision weights: EXACT MATCH

[5] FREEZE MODEL
Parameters: 7566219264
Trainable: 0

[6] INITIALIZE IMAGE PROCESSOR
Vision patches: 576 — PASS

[7] GPU MEMORY
GPU 0: 7.599 GiB allocated
GPU 1: 6.994 GiB allocated

Saved: /kaggle/working/B2_FastV/configs/model_load_report.json

CELL 4 MODEL VERIFICATION PASSED

[3] PREPARE MULTIMODAL INPUT
Image tensor: (1, 3, 336, 336)
Input token positions: 22
Multimodal sequence length: 597
Visual start: 5
Visual tokens: 576

[4] UNPRUNED NATIVE EQUIVALENCE
Maximum absolute logit error: 0.00390625
Greedy next token: MA

In [12]:

# ============================================================
# B2 — CELL 10: COMPLETE-ANSWER VALIDATION
# Native 100% correctness control: 8 greedy steps
# B2 experiments: 75%, 50%, 25% only
# ============================================================

from pathlib import Path
import os
import subprocess
import textwrap
import json

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"
SCRIPT = ROOT / "scripts/cell10_full_answer_validation.py"
OUTPUT = ROOT / "predictions/cell10_full_answer_smoke.json"

for required in (
    PYTHON,
    ROOT / "load_b2_model.py",
    ROOT / "fastv_cached.py",
    ROOT / "configs/real_7b_fastv_smoke_test.json",
    ROOT / "configs/experiment_protocol.json",
):
    assert required.is_file(), f"Missing: {required}"

print("=" * 76)
print("B2 — CELL 10: FULL-ANSWER VALIDATION")
print("=" * 76)

code = r'''
import gc
import json
import sys
from pathlib import Path

import torch
from PIL import Image
from transformers.cache_utils import DynamicCache

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(REPO))

from load_b2_model import load_b2_model
from fastv_cached import (
    fastv_cached_prefill,
    fastv_cached_decode_step,
)
from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
)
from llava.conversation import conv_templates
from llava.mm_utils import (
    tokenizer_image_token,
    process_images,
)

MAX_NEW_TOKENS = 64
IDENTITY_STEPS = 8
RETENTIONS = (75, 50, 25)

torch.set_grad_enabled(False)

print("=" * 76, flush=True)
print("B2 — FULL ANSWER GENERATION TEST", flush=True)
print("=" * 76, flush=True)

# ------------------------------------------------------------
# 1. Load the same verified SLAKE example
# ------------------------------------------------------------

print("\n[1] LOAD SLAKE QUESTION", flush=True)

protocol = json.loads(
    (ROOT / "configs/experiment_protocol.json")
    .read_text(encoding="utf-8")
)

assert tuple(protocol["retention_percentages"]) == RETENTIONS
assert protocol["decoding"]["max_new_tokens"] == MAX_NEW_TOKENS

with open(protocol["test_json"], encoding="utf-8") as f:
    records = json.load(f)

english = [
    r for r in records
    if str(r.get("q_lang", "")).strip().lower() == "en"
]
assert len(english) == 1061

sample = next(
    r for r in english
    if str(r["qid"]) == "11934"
)

image_path = Path(protocol["images_root"]) / sample["img_name"]
assert image_path.is_file()

print("QID:", sample["qid"], flush=True)
print("Question:", sample["question"], flush=True)
print("Reference:", sample["answer"], flush=True)

# ------------------------------------------------------------
# 2. Load frozen model
# ------------------------------------------------------------

print("\n[2] LOAD FROZEN LLAVA-MED", flush=True)

tokenizer, model, vision_tower = load_b2_model()

assert not model.training
assert all(not p.requires_grad for p in model.parameters())

eos_id = tokenizer.eos_token_id
assert eos_id is not None

# ------------------------------------------------------------
# 3. Prepare multimodal embeddings
# ------------------------------------------------------------

print("\n[3] PREPARE MULTIMODAL INPUT", flush=True)

conversation = conv_templates["mistral_instruct"].copy()

conversation.append_message(
    conversation.roles[0],
    DEFAULT_IMAGE_TOKEN + "\n" + sample["question"],
)
conversation.append_message(conversation.roles[1], None)

input_ids = tokenizer_image_token(
    conversation.get_prompt(),
    tokenizer,
    IMAGE_TOKEN_INDEX,
    return_tensors="pt",
).unsqueeze(0).to("cuda:0")

image_positions = (
    (input_ids[0] == IMAGE_TOKEN_INDEX)
    .nonzero(as_tuple=True)[0]
)

assert image_positions.numel() == 1
visual_start = int(image_positions[0].item())

with Image.open(image_path) as image:
    image_rgb = image.convert("RGB")

pixels = process_images(
    [image_rgb],
    vision_tower.image_processor,
    model.config,
)

if isinstance(pixels, list):
    assert len(pixels) == 1
    pixels = pixels[0].unsqueeze(0)

pixels = pixels.to(
    device="cuda:0",
    dtype=torch.float16,
)

with torch.inference_mode():
    prepared = model.prepare_inputs_labels_for_multimodal(
        input_ids=input_ids,
        position_ids=None,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        images=pixels,
    )

embeddings = prepared[4]

assert embeddings is not None

full_length = int(embeddings.shape[1])
assert full_length - input_ids.shape[1] + 1 == 576
assert visual_start + 576 <= full_length

print("Full sequence length:", full_length, flush=True)
print("Visual span start:", visual_start, flush=True)

# ------------------------------------------------------------
# 4. Native vs custom unpruned generation identity
# ------------------------------------------------------------

print("\n[4] EIGHT-STEP NATIVE IDENTITY", flush=True)

native_cache = DynamicCache()

with torch.inference_mode():
    native_output = model(
        inputs_embeds=embeddings,
        attention_mask=torch.ones(
            (1, full_length),
            dtype=torch.long,
            device="cuda:0",
        ),
        past_key_values=native_cache,
        use_cache=True,
        return_dict=True,
    )

native_logits = native_output.logits[:, -1, :].float()

state = fastv_cached_prefill(
    model,
    embeddings,
    visual_start=visual_start,
    retention_pct=100,
    allow_unpruned_control=True,
)

custom_logits = state["last_logits"]

identity_tokens = []
max_logit_error = 0.0

for step in range(IDENTITY_STEPS):

    # Keep comparisons on one GPU; never subtract
    # tensors residing on separate devices.
    comparison = custom_logits.to(native_logits.device)

    error = (
        native_logits - comparison
    ).abs().max().item()

    max_logit_error = max(max_logit_error, error)

    assert torch.allclose(
        native_logits,
        comparison,
        atol=2e-2,
        rtol=2e-3,
    ), f"Logits differ at step {step + 1}: {error}"

    native_token_id = int(
        native_logits.argmax(dim=-1).item()
    )
    custom_token_id = int(
        comparison.argmax(dim=-1).item()
    )

    assert native_token_id == custom_token_id, (
        f"Greedy token mismatch at step {step + 1}"
    )

    identity_tokens.append(native_token_id)

    print(
        f"Step {step+1}: "
        f"token={native_token_id}, "
        f"logit_max_diff={error:.6f} — PASS",
        flush=True,
    )

    # Stop correctly at EOS and avoid processing a
    # token after the last requested comparison.
    if native_token_id == eos_id:
        print("EOS reached during identity check.", flush=True)
        break

    if step == IDENTITY_STEPS - 1:
        break

    token = torch.tensor(
        [[native_token_id]],
        dtype=torch.long,
        device="cuda:0",
    )

    with torch.inference_mode():
        native_next = model(
            input_ids=token,
            attention_mask=torch.ones(
                (1, full_length + step + 1),
                dtype=torch.long,
                device="cuda:0",
            ),
            position_ids=torch.tensor(
                [[full_length + step]],
                dtype=torch.long,
                device="cuda:0",
            ),
            past_key_values=native_cache,
            use_cache=True,
            return_dict=True,
        )

    native_logits = native_next.logits[:, -1, :].float()

    custom_logits, state = fastv_cached_decode_step(
        model,
        token,
        state,
    )

print("Native/custom identity: PASS", flush=True)

del native_cache, native_output, state
del native_logits, custom_logits
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Full greedy generation at three retentions
# ------------------------------------------------------------

print("\n[5] FULL FASTV ANSWERS", flush=True)

results = []

for pct in RETENTIONS:

    print(f"\n--- B2 FASTV {pct}% ---", flush=True)

    state = fastv_cached_prefill(
        model,
        embeddings,
        visual_start=visual_start,
        retention_pct=pct,
        pruning_after_layer=2,
    )

    reduced_length = full_length - 576 + (576 * pct // 100)

    assert state["prefill_cache_lengths"] == (
        [full_length] * 2
        + [reduced_length] * 30
    )

    logits = state["last_logits"]
    generated = []
    eos_terminated = False

    for step in range(MAX_NEW_TOKENS):

        next_id = int(logits.argmax(dim=-1).item())
        generated.append(next_id)

        if next_id == eos_id:
            eos_terminated = True
            break

        if step == MAX_NEW_TOKENS - 1:
            break

        token = torch.tensor(
            [[next_id]],
            device="cuda:0",
            dtype=torch.long,
        )

        logits, state = fastv_cached_decode_step(
            model,
            token,
            state,
        )

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    expected_cache = (
        [full_length + state["generated_count"]] * 2
        + [reduced_length + state["generated_count"]] * 30
    )

    actual_cache = [
        int(state["cache"].get_seq_length(i))
        for i in range(32)
    ]

    assert actual_cache == expected_cache
    assert 1 <= len(generated) <= MAX_NEW_TOKENS

    print("Visual tokens:", 576 * pct // 100, flush=True)
    print("Generated tokens:", len(generated), flush=True)
    print("Stopped on EOS:", eos_terminated, flush=True)
    print("Answer:", repr(prediction), flush=True)
    print("Cache consistency: PASS", flush=True)

    results.append({
        "qid": str(sample["qid"]),
        "retention_pct": pct,
        "retained_visual_tokens": 576 * pct // 100,
        "original_sequence_length": full_length,
        "reduced_sequence_length": reduced_length,
        "prediction": prediction,
        "generated_token_ids": generated,
        "generated_token_count": len(generated),
        "eos_terminated": eos_terminated,
        "reached_generation_limit": (
            len(generated) == MAX_NEW_TOKENS
            and not eos_terminated
        ),
        "prefill_cache_lengths": state["prefill_cache_lengths"],
        "final_cache_lengths": actual_cache,
    })

    del state, logits
    gc.collect()
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6. Save only after successful validation
# ------------------------------------------------------------

print("\n[6] SAVE CELL 10 RESULTS", flush=True)

output = ROOT / "predictions/cell10_full_answer_smoke.json"
output.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "model": protocol["model_id"],
    "qid": str(sample["qid"]),
    "question": sample["question"],
    "reference": sample["answer"],
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "native_control_tokens": identity_tokens,
    "native_control_max_logit_error": max_logit_error,
    "retentions": [75, 50, 25],
    "results": results,
    "smoke_test_only": True,
}

assert [r["retention_pct"] for r in results] == list(RETENTIONS)

output.write_text(
    json.dumps(payload, indent=2),
    encoding="utf-8",
)

print("Saved:", output, flush=True)

print("\n" + "=" * 76, flush=True)
print("CELL 10 PASSED — FULL ANSWER GENERATION VERIFIED", flush=True)
print("=" * 76, flush=True)
'''

SCRIPT.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_text(textwrap.dedent(code).lstrip(), encoding="utf-8")

print("\n[1] CREATED VALIDATION SCRIPT:", SCRIPT)

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)
print("[2] Syntax: PASS")

print("\n[3] RUN FULL-ANSWER VALIDATION\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])
env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(SCRIPT)],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 10 validation failed. "
        "Share the traceback before proceeding."
    )

assert OUTPUT.is_file()
payload = json.loads(OUTPUT.read_text(encoding="utf-8"))

assert payload["status"] == "PASS"
assert len(payload["results"]) == 3

print("\n" + "=" * 76)
print("B2 — CELL 10 COMPLETE")
print("=" * 76)
print("Native generation control: PASS")
print("FastV 75%, 50%, 25%: PASS")
print("Output:", OUTPUT)


B2 — CELL 10: FULL-ANSWER VALIDATION

[1] CREATED VALIDATION SCRIPT: /kaggle/working/B2_FastV/scripts/cell10_full_answer_validation.py
[2] Syntax: PASS

[3] RUN FULL-ANSWER VALIDATION

B2 — FULL ANSWER GENERATION TEST

[1] LOAD SLAKE QUESTION
QID: 11934
Question: What modality is used to take this image?
Reference: CT

[2] LOAD FROZEN LLAVA-MED
B2 — FROZEN TWO-GPU LLAVA-MED LOAD

[1] TOKENIZER
Vocabulary: 32000

[2] LOAD ORIGINAL CHECKPOINT


Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.02s/it]


missing_keys: 0
unexpected_keys: 0
mismatched_keys: 0
error_msgs: 0

[3] VERIFY GPU PLACEMENT
model.embed_tokens: cuda:0 — PASS
model.vision_tower: cuda:0 — PASS
model.mm_projector: cuda:0 — PASS
model.layers.0: cuda:0 — PASS
model.layers.15: cuda:0 — PASS
model.layers.16: cuda:1 — PASS
model.layers.31: cuda:1 — PASS
model.norm: cuda:1 — PASS
lm_head: cuda:1 — PASS

[4] VERIFY VISION WEIGHTS
Vision weight max absolute error: 0.0
Original vision weights: EXACT MATCH

[5] FREEZE MODEL
Parameters: 7566219264
Trainable: 0

[6] INITIALIZE IMAGE PROCESSOR
Vision patches: 576 — PASS

[7] GPU MEMORY
GPU 0: 7.599 GiB allocated
GPU 1: 6.994 GiB allocated

Saved: /kaggle/working/B2_FastV/configs/model_load_report.json

CELL 4 MODEL VERIFICATION PASSED

[3] PREPARE MULTIMODAL INPUT
Full sequence length: 597
Visual span start: 5

[4] EIGHT-STEP NATIVE IDENTITY
Step 1: token=415, logit_max_diff=0.003906 — PASS
Step 2: token=3469, logit_max_diff=0.000000 — PASS
Step 3: token=349, logit_max_diff=0.000

In [13]:

# ============================================================
# B2 — CELL 11: OFFICIAL SLAKE MANIFEST + B1-COMPATIBLE METRICS
#
# Creates:
#   manifests/b2_slake_official_test_english.jsonl
#   manifests/b2_fastv_3183_manifest.jsonl
#   b2_metrics.py
#   configs/b1_metric_compatibility_test.json
#   configs/b2_manifest_report.json
#
# No GPU, no model loading, no inference.
# ============================================================

from pathlib import Path
from collections import Counter
import json
import re
import importlib.util
import textwrap

ROOT = Path("/kaggle/working/B2_FastV")
CONFIG = ROOT / "configs/experiment_protocol.json"

assert CONFIG.is_file(), "Cell 1 configuration missing."
assert (
    ROOT / "predictions/cell10_full_answer_smoke.json"
).is_file(), "Cell 10 must pass first."

protocol = json.loads(CONFIG.read_text(encoding="utf-8"))

assert protocol["retention_percentages"] == [75, 50, 25]
assert protocol["expected_predictions"] == 3183

OUT = ROOT / "manifests"
OUT.mkdir(parents=True, exist_ok=True)

QUESTION_MANIFEST = OUT / "b2_slake_official_test_english.jsonl"
FULL_MANIFEST = OUT / "b2_fastv_3183_manifest.jsonl"
METRICS_MODULE = ROOT / "b2_metrics.py"
METRICS_REPORT = ROOT / "configs/b1_metric_compatibility_test.json"
MANIFEST_REPORT = ROOT / "configs/b2_manifest_report.json"

print("=" * 76)
print("B2 — CELL 11: OFFICIAL MANIFEST AND B1 METRICS")
print("=" * 76)

# ------------------------------------------------------------
# 1. Load official SLAKE test data
# ------------------------------------------------------------

print("\n[1] LOAD OFFICIAL SLAKE TEST DATA")

TEST_JSON = Path(protocol["test_json"])
IMAGE_ROOT = Path(protocol["images_root"])

assert TEST_JSON.is_file()
assert IMAGE_ROOT.is_dir()

with TEST_JSON.open(encoding="utf-8") as f:
    raw_records = json.load(f)

assert len(raw_records) == 2094

english = [
    (original_index, row)
    for original_index, row in enumerate(raw_records)
    if str(row.get("q_lang", "")).strip().lower() == "en"
]

assert len(english) == 1061

# ------------------------------------------------------------
# 2. Construct original-order question manifest
# ------------------------------------------------------------

print("\n[2] BUILD QUESTION MANIFEST")

questions = []

for original_index, row in english:

    qid = str(row["qid"])
    answer_type = str(row["answer_type"]).strip().lower()

    assert answer_type in ("closed", "open")

    question = str(row["question"])
    reference = str(row["answer"])
    img_name = str(row["img_name"])

    assert question.strip()
    assert reference.strip()

    image_path = IMAGE_ROOT / img_name
    assert image_path.is_file(), image_path

    questions.append({
        "qid": qid,
        "original_test_index": original_index,
        "question": question,
        "reference": reference,
        "answer": reference,
        "answer_type": answer_type,
        "img_name": img_name,
        "q_lang": "en",
    })

assert len(questions) == 1061
assert len({r["qid"] for r in questions}) == 1061

counts = Counter(r["answer_type"] for r in questions)

assert counts == {"closed": 416, "open": 645}

# Atomic replacement prevents a partially written
# manifest from being treated as a completed one.
def write_jsonl_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    temporary.replace(path)

write_jsonl_atomic(QUESTION_MANIFEST, questions)

print("Questions:", len(questions))
print("CLOSED:", counts["closed"])
print("OPEN:", counts["open"])
print("Saved:", QUESTION_MANIFEST)

# ------------------------------------------------------------
# 3. Expand question manifest for FastV retentions
# ------------------------------------------------------------

print("\n[3] BUILD 3,183-PREDICTION MANIFEST")

retentions = (75, 50, 25)
expanded = []

for record in questions:
    for retention in retentions:
        expanded.append({
            **record,
            "retention_pct": retention,
            "original_visual_tokens": 576,
            "retained_visual_tokens": 576 * retention // 100,
            "pruning_after_layer": 2,
        })

assert len(expanded) == 3183

pairs = [
    (r["qid"], r["retention_pct"])
    for r in expanded
]

assert len(pairs) == len(set(pairs))

for pct, count in ((75, 432), (50, 288), (25, 144)):
    rows = [
        r for r in expanded
        if r["retention_pct"] == pct
    ]

    assert len(rows) == 1061
    assert all(
        r["retained_visual_tokens"] == count
        for r in rows
    )

write_jsonl_atomic(FULL_MANIFEST, expanded)

print("Retention levels:", retentions)
print("Expected predictions:", len(expanded))
print("Unique QID/retention combinations:", len(set(pairs)))
print("Saved:", FULL_MANIFEST)

# ------------------------------------------------------------
# 4. Install exact B1 Cell 11B scoring functions
# ------------------------------------------------------------

print("\n[4] INSTALL B1-COMPATIBLE SCORER")

metrics_code = r'''
import re
from collections import Counter


def normalize_answer(text):
    text = str(text).lower().strip()
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


def parse_yes_no(text):
    normalized = normalize_answer(text)

    match = re.match(
        r"^(yes|no)(?:\s|$)",
        normalized
    )

    return match.group(1) if match else None


def closed_accuracy(reference, prediction):
    """
    Closed-ended SLAKE accuracy.

    Primary metric:
      normalized exact match.

    For Yes/No references, permit a leading Yes/No
    followed by explanatory text.

    For categorical references, use strict normalized
    exact matching, not substring matching.
    """

    gold = normalize_answer(reference)
    pred = normalize_answer(prediction)

    if not gold:
        raise ValueError("Empty closed-ended reference")

    if gold in ("yes", "no"):
        extracted = parse_yes_no(prediction)

        correct = int(extracted == gold)

        method = "yes_no_extraction"

    else:
        extracted = None
        correct = int(pred == gold)

        method = "categorical_exact_match"

    return {
        "correct": correct,
        "method": method,
        "normalized_reference": gold,
        "normalized_prediction": pred,
        "yes_no_prediction": extracted
    }


def open_token_metrics(reference, prediction):
    gold_tokens = normalize_answer(reference).split()
    pred_tokens = normalize_answer(prediction).split()

    if not gold_tokens:
        raise ValueError("Empty open-ended reference")

    overlap = sum(
        (Counter(gold_tokens) & Counter(pred_tokens)).values()
    )

    precision = (
        overlap / len(pred_tokens)
        if pred_tokens else 0.0
    )

    recall = overlap / len(gold_tokens)

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": int(
            normalize_answer(reference)
            == normalize_answer(prediction)
        )
    }


def evaluate_answer(answer_type, reference, prediction):
    answer_type = str(answer_type).lower().strip()

    if answer_type == "closed":
        result = closed_accuracy(reference, prediction)

        return {
            "closed_accuracy": result["correct"],
            "closed_scoring_method": result["method"],
            "open_precision": None,
            "open_recall": None,
            "open_f1": None
        }

    if answer_type == "open":
        result = open_token_metrics(reference, prediction)

        return {
            "closed_accuracy": None,
            "closed_scoring_method": None,
            "open_precision": result["precision"],
            "open_recall": result["recall"],
            "open_f1": result["f1"]
        }

    raise ValueError(f"Unsupported answer type: {answer_type}")
'''

METRICS_MODULE.write_text(
    textwrap.dedent(metrics_code).lstrip(),
    encoding="utf-8"
)

spec = importlib.util.spec_from_file_location(
    "b2_metrics", METRICS_MODULE
)

metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)

print("Saved:", METRICS_MODULE)

# ------------------------------------------------------------
# 5. Unit-test exact B1 scoring behavior
# ------------------------------------------------------------

print("\n[5] VERIFY CLOSED-ANSWER SCORING")

closed_tests = [
    ("No", "No, the image does not contain the liver.", 1),
    ("Yes", "Yes, it is visible.", 1),
    ("Yes", "No, it is not visible.", 0),
    ("Lung", "Lung", 1),
    ("Lung", "The lung is visible.", 0),
    ("Heart", "Lung", 0),
    ("Lung", "Lung and heart", 0),
    ("CT", "The image is taken using a computed tomography (CT) scan.", 0),
]

for reference, prediction, expected in closed_tests:

    actual = metrics.closed_accuracy(
        reference, prediction
    )["correct"]

    assert actual == expected, (
        reference, prediction, actual, expected
    )

    print(
        f"Reference={reference!r} | "
        f"Correct={actual} — PASS"
    )

print("\n[6] VERIFY OPEN-ANSWER SCORING")

open_tests = [
    ("heart", "heart", 1.0, 1.0),
    ("heart lung", "heart", 0.5, 2 / 3),
    ("heart", "kidney", 0.0, 0.0),
    ("red-red", "red", 0.5, 2 / 3),
]

for reference, prediction, expected_recall, expected_f1 in open_tests:

    result = metrics.open_token_metrics(
        reference, prediction
    )

    assert abs(
        result["recall"] - expected_recall
    ) < 1e-12

    assert abs(
        result["f1"] - expected_f1
    ) < 1e-12

    print(
        f"Reference={reference!r} | "
        f"Recall={result['recall']:.4f} | "
        f"F1={result['f1']:.4f} — PASS"
    )

# ------------------------------------------------------------
# 7. Verify the scorer on all ground-truth answers
# ------------------------------------------------------------

print("\n[7] FULL-DATASET SCORER SELF-TEST")

for record in questions:

    result = metrics.evaluate_answer(
        record["answer_type"],
        record["reference"],
        record["reference"],
    )

    if record["answer_type"] == "closed":
        assert result["closed_accuracy"] == 1
    else:
        assert abs(result["open_recall"] - 1.0) < 1e-12
        assert abs(result["open_f1"] - 1.0) < 1e-12

print("Ground-truth self-matches:", len(questions), "— PASS")

# ------------------------------------------------------------
# 8. Save validation reports
# ------------------------------------------------------------

print("\n[8] SAVE VERIFICATION REPORTS")

metric_report = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "source": "B1 Cell 11B",
    "source_notebook": (
        "baselines/b1_results/b1-benchmark-final.ipynb"
    ),
    "module": str(METRICS_MODULE),
    "normalization": "B1 normalize_answer",
    "closed_yes_no": "B1 leading yes/no extraction",
    "closed_categorical": "B1 normalized exact match",
    "open_scoring": "B1 Counter token-overlap precision recall F1",
    "closed_unit_tests": len(closed_tests),
    "open_unit_tests": len(open_tests),
    "ground_truth_self_tests": len(questions),
    "semantic_review": False,
    "llm_judge": False,
}

manifest_report = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "source_test_json": str(TEST_JSON),
    "question_manifest": str(QUESTION_MANIFEST),
    "expanded_manifest": str(FULL_MANIFEST),
    "language": "en",
    "split": "official_test",
    "questions": 1061,
    "closed_questions": 416,
    "open_questions": 645,
    "retentions": [75, 50, 25],
    "retained_visual_tokens": {
        "75": 432,
        "50": 288,
        "25": 144,
    },
    "total_predictions": 3183,
    "metrics_module": str(METRICS_MODULE),
}

METRICS_REPORT.write_text(
    json.dumps(metric_report, indent=2),
    encoding="utf-8"
)

MANIFEST_REPORT.write_text(
    json.dumps(manifest_report, indent=2),
    encoding="utf-8"
)

print("Metrics report:", METRICS_REPORT)
print("Manifest report:", MANIFEST_REPORT)

print("\n" + "=" * 76)
print("CELL 11 PASSED — MANIFEST AND B1 METRICS VERIFIED")
print("=" * 76)
print("SLAKE English questions: 1061")
print("CLOSED: 416 | OPEN: 645")
print("FastV retentions: 75%, 50%, 25%")
print("Total predictions: 3183")
print("B1-compatible evaluation: PASS")


B2 — CELL 11: OFFICIAL MANIFEST AND B1 METRICS

[1] LOAD OFFICIAL SLAKE TEST DATA

[2] BUILD QUESTION MANIFEST
Questions: 1061
CLOSED: 416
OPEN: 645
Saved: /kaggle/working/B2_FastV/manifests/b2_slake_official_test_english.jsonl

[3] BUILD 3,183-PREDICTION MANIFEST
Retention levels: (75, 50, 25)
Expected predictions: 3183
Unique QID/retention combinations: 3183
Saved: /kaggle/working/B2_FastV/manifests/b2_fastv_3183_manifest.jsonl

[4] INSTALL B1-COMPATIBLE SCORER
Saved: /kaggle/working/B2_FastV/b2_metrics.py

[5] VERIFY CLOSED-ANSWER SCORING
Reference='No' | Correct=1 — PASS
Reference='Yes' | Correct=1 — PASS
Reference='Yes' | Correct=0 — PASS
Reference='Lung' | Correct=1 — PASS
Reference='Lung' | Correct=0 — PASS
Reference='Heart' | Correct=0 — PASS
Reference='Lung' | Correct=0 — PASS
Reference='CT' | Correct=0 — PASS

[6] VERIFY OPEN-ANSWER SCORING
Reference='heart' | Recall=1.0000 | F1=1.0000 — PASS
Reference='heart lung' | Recall=0.5000 | F1=0.6667 — PASS
Reference='heart' | Recall

In [14]:

# ============================================================
# B2 — CELL 12: RESUMABLE FASTV INFERENCE RUNNER
#
# Pilot: 2 SLAKE questions x 3 retentions = 6 predictions
# Full run will be launched separately after verification.
# ============================================================

from pathlib import Path
import os
import subprocess
import textwrap
import json

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

RUNNER = ROOT / "scripts/b2_fastv_inference_runner.py"
PILOT = ROOT / "predictions/b2_fastv_pilot_predictions.jsonl"
PILOT_REPORT = ROOT / "configs/b2_pilot_report.json"

required = [
    PYTHON,
    ROOT / "load_b2_model.py",
    ROOT / "fastv_cached.py",
    ROOT / "b2_metrics.py",
    ROOT / "manifests/b2_slake_official_test_english.jsonl",
    ROOT / "manifests/b2_fastv_3183_manifest.jsonl",
    ROOT / "configs/b1_metric_compatibility_test.json",
    ROOT / "predictions/cell10_full_answer_smoke.json",
]

for path in required:
    assert path.is_file(), f"Missing prerequisite: {path}"

print("=" * 76)
print("B2 — CELL 12: RESUMABLE INFERENCE RUNNER")
print("=" * 76)

runner_code = r'''
import argparse
import gc
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

import torch
from PIL import Image

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(REPO))

from load_b2_model import load_b2_model
from fastv_cached import (
    fastv_cached_prefill,
    fastv_cached_decode_step,
)
from b2_metrics import evaluate_answer

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
)
from llava.conversation import conv_templates
from llava.mm_utils import (
    tokenizer_image_token,
    process_images,
)

RETENTIONS = (75, 50, 25)
MAX_NEW_TOKENS = 64
EXPECTED_VISUAL = {75: 432, 50: 288, 25: 144}

torch.set_grad_enabled(False)


def sync_gpus():
    for gpu in (0, 1):
        torch.cuda.synchronize(gpu)


def read_jsonl(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


def key_for(record):
    return (
        str(record["qid"]),
        int(record["retention_pct"]),
    )


def verify_existing(path, allowed_keys):
    """
    Validate the existing checkpoint before resuming.
    Never silently accept duplicate or foreign records.
    """
    rows = read_jsonl(path)
    completed = {}

    for record in rows:
        key = key_for(record)

        assert key in allowed_keys, (
            f"Unexpected checkpoint record: {key}"
        )

        assert key not in completed, (
            f"Duplicate checkpoint record: {key}"
        )

        assert "prediction" in record
        assert "scoring" in record
        assert record["retained_visual_tokens"] == (
            EXPECTED_VISUAL[key[1]]
        )

        completed[key] = record

    return completed


def prepare_question(sample, tokenizer, model, tower, image_root):
    """
    Prepare multimodal embeddings once per question,
    then reuse them for the three retention settings.
    """
    conv = conv_templates["mistral_instruct"].copy()

    conv.append_message(
        conv.roles[0],
        DEFAULT_IMAGE_TOKEN + "\n" + sample["question"],
    )
    conv.append_message(conv.roles[1], None)

    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
    ).unsqueeze(0).to("cuda:0")

    positions = (
        (input_ids[0] == IMAGE_TOKEN_INDEX)
        .nonzero(as_tuple=True)[0]
    )

    assert positions.numel() == 1
    visual_start = int(positions[0].item())

    image_path = image_root / sample["img_name"]
    assert image_path.is_file(), image_path

    sync_gpus()
    image_begin = time.perf_counter()

    with Image.open(image_path) as image:
        rgb = image.convert("RGB")

    pixels = process_images(
        [rgb],
        tower.image_processor,
        model.config,
    )

    if isinstance(pixels, list):
        assert len(pixels) == 1
        pixels = pixels[0].unsqueeze(0)

    pixels = pixels.to(
        "cuda:0",
        dtype=torch.float16,
    )

    with torch.inference_mode():
        prepared = model.prepare_inputs_labels_for_multimodal(
            input_ids=input_ids,
            position_ids=None,
            attention_mask=None,
            past_key_values=None,
            labels=None,
            images=pixels,
        )

    embeddings = prepared[4]
    assert embeddings is not None

    sync_gpus()
    image_ms = (
        time.perf_counter() - image_begin
    ) * 1000.0

    full_length = int(embeddings.shape[1])

    assert full_length - input_ids.shape[1] + 1 == 576

    return {
        "embeddings": embeddings,
        "visual_start": visual_start,
        "full_length": full_length,
        "image_encoding_ms": image_ms,
    }


def infer_one(model, tokenizer, prepared, pct):
    """
    Measure prefill and autoregressive decode separately.

    CUDA synchronization makes elapsed wall-clock
    measurements include completed GPU execution.
    """
    full_length = prepared["full_length"]
    reduced_length = (
        full_length - 576 + EXPECTED_VISUAL[pct]
    )

    sync_gpus()

    # These are per-inference allocation peaks. Since
    # GPUs are reset/read at separate instants, their
    # sum is NOT a simultaneous total GPU peak.
    for gpu in (0, 1):
        torch.cuda.reset_peak_memory_stats(gpu)

    inference_begin = time.perf_counter()

    prefill_begin = time.perf_counter()

    state = fastv_cached_prefill(
        model,
        prepared["embeddings"],
        visual_start=prepared["visual_start"],
        retention_pct=pct,
        visual_count=576,
        pruning_after_layer=2,
    )

    sync_gpus()

    prefill_ms = (
        time.perf_counter() - prefill_begin
    ) * 1000.0

    expected_cache = (
        [full_length] * 2
        + [reduced_length] * 30
    )

    assert state["prefill_cache_lengths"] == expected_cache

    logits = state["last_logits"]
    generated = []
    eos_terminated = False

    decode_begin = time.perf_counter()

    for step in range(MAX_NEW_TOKENS):
        token_id = int(
            logits.argmax(dim=-1).item()
        )

        generated.append(token_id)

        if token_id == tokenizer.eos_token_id:
            eos_terminated = True
            break

        if step == MAX_NEW_TOKENS - 1:
            break

        token = torch.tensor(
            [[token_id]],
            device="cuda:0",
            dtype=torch.long,
        )

        logits, state = fastv_cached_decode_step(
            model, token, state
        )

    sync_gpus()

    decode_ms = (
        time.perf_counter() - decode_begin
    ) * 1000.0

    total_ms = (
        time.perf_counter() - inference_begin
    ) * 1000.0

    final_cache = [
        int(state["cache"].get_seq_length(i))
        for i in range(32)
    ]

    expected_final = (
        [full_length + state["generated_count"]] * 2
        + [reduced_length + state["generated_count"]] * 30
    )

    assert final_cache == expected_final
    assert 1 <= len(generated) <= MAX_NEW_TOKENS

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    memory = {}

    for gpu in (0, 1):
        memory[f"gpu{gpu}_peak_allocated_gib"] = (
            torch.cuda.max_memory_allocated(gpu)
            / 1024**3
        )
        memory[f"gpu{gpu}_peak_reserved_gib"] = (
            torch.cuda.max_memory_reserved(gpu)
            / 1024**3
        )

    result = {
        "prediction": prediction,
        "generated_token_ids": generated,
        "generated_token_count": len(generated),
        "eos_terminated": eos_terminated,
        "reached_generation_limit": (
            len(generated) == MAX_NEW_TOKENS
            and not eos_terminated
        ),
        "original_sequence_length": full_length,
        "reduced_sequence_length": reduced_length,
        "prefill_cache_lengths": expected_cache,
        "final_cache_lengths": final_cache,
        "profiling": {
            "llm_prefill_ms": prefill_ms,
            "decode_ms": decode_ms,
            "total_inference_ms": total_ms,
            "measurement": "synchronized_wall_clock",
            "image_encoding_excluded": True,
            **memory,
        },
    }

    del state, logits
    return result


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--mode",
        choices=["pilot", "full"],
        required=True,
    )

    args = parser.parse_args()

    protocol = json.loads(
        (ROOT / "configs/experiment_protocol.json")
        .read_text(encoding="utf-8")
    )

    assert protocol["retention_percentages"] == [
        75, 50, 25
    ]
    assert protocol["decoding"]["max_new_tokens"] == 64

    question_manifest = (
        ROOT / "manifests/"
        "b2_slake_official_test_english.jsonl"
    )

    expanded_manifest = (
        ROOT / "manifests/"
        "b2_fastv_3183_manifest.jsonl"
    )

    questions = read_jsonl(question_manifest)
    expanded = read_jsonl(expanded_manifest)

    assert len(questions) == 1061
    assert len(expanded) == 3183

    if args.mode == "pilot":
        selected = questions[:2]
        output = (
            ROOT / "predictions/"
            "b2_fastv_pilot_predictions.jsonl"
        )
    else:
        selected = questions
        output = (
            ROOT / "predictions/"
            "b2_fastv_full_predictions.jsonl"
        )

    expected_keys = {
        (str(q["qid"]), pct)
        for q in selected
        for pct in RETENTIONS
    }

    assert len(expected_keys) == len(selected) * 3

    # Verify that every requested combination
    # appears in the locked expanded manifest.
    manifest_keys = {
        key_for(r) for r in expanded
    }
    assert expected_keys.issubset(manifest_keys)

    output.parent.mkdir(parents=True, exist_ok=True)

    completed = verify_existing(
        output, expected_keys
    )

    pending = expected_keys - set(completed)

    print("=" * 76, flush=True)
    print("B2 — FASTV CHECKPOINTED INFERENCE", flush=True)
    print("=" * 76, flush=True)
    print("Mode:", args.mode, flush=True)
    print("Questions:", len(selected), flush=True)
    print("Retentions:", RETENTIONS, flush=True)
    print("Expected predictions:", len(expected_keys), flush=True)
    print("Already completed:", len(completed), flush=True)
    print("Remaining:", len(pending), flush=True)
    print("Checkpoint:", output, flush=True)

    if pending:
        print("\n[1] LOAD FROZEN MODEL", flush=True)
        tokenizer, model, tower = load_b2_model()

        assert not model.training
        assert all(
            not p.requires_grad
            for p in model.parameters()
        )

        image_root = Path(protocol["images_root"])

        print("\n[2] RUN INFERENCE", flush=True)

        with output.open(
            "a", encoding="utf-8"
        ) as out:

            for q_index, question in enumerate(
                selected, start=1
            ):
                qid = str(question["qid"])

                remaining_pcts = [
                    pct for pct in RETENTIONS
                    if (qid, pct) not in completed
                ]

                if not remaining_pcts:
                    continue

                print(
                    f"\nQuestion {q_index}/{len(selected)} "
                    f"| QID={qid} "
                    f"| pending={remaining_pcts}",
                    flush=True,
                )

                prepared = prepare_question(
                    question,
                    tokenizer,
                    model,
                    tower,
                    image_root,
                )

                for pct in remaining_pcts:

                    inference = infer_one(
                        model,
                        tokenizer,
                        prepared,
                        pct,
                    )

                    scoring = evaluate_answer(
                        question["answer_type"],
                        question["reference"],
                        inference["prediction"],
                    )

                    record = {
                        "experiment": "B2_FastV",
                        "qid": qid,
                        "original_test_index": (
                            question["original_test_index"]
                        ),
                        "question": question["question"],
                        "reference": question["reference"],
                        "answer_type": question["answer_type"],
                        "img_name": question["img_name"],
                        "retention_pct": pct,
                        "original_visual_tokens": 576,
                        "retained_visual_tokens": (
                            EXPECTED_VISUAL[pct]
                        ),
                        "pruning_after_layer": 2,
                        "image_encoding_ms": (
                            prepared["image_encoding_ms"]
                        ),
                        "scoring": scoring,
                        **inference,
                    }

                    key = key_for(record)
                    assert key not in completed

                    out.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        ) + "\n"
                    )
                    out.flush()
                    os.fsync(out.fileno())

                    completed[key] = record

                    profile = record["profiling"]

                    print(
                        f"  {pct}% | "
                        f"{record['generated_token_count']} tokens "
                        f"| prefill={profile['llm_prefill_ms']:.2f}ms "
                        f"| decode={profile['decode_ms']:.2f}ms",
                        flush=True,
                    )

                    print(
                        "  Answer:",
                        repr(record["prediction"]),
                        flush=True,
                    )
                    print(
                        f"  Saved {len(completed)}/"
                        f"{len(expected_keys)}",
                        flush=True,
                    )

                del prepared

                # Periodically collect Python garbage,
                # without changing frozen model weights.
                if q_index % 25 == 0:
                    gc.collect()

    # Re-read from disk to verify durable output.
    completed = verify_existing(
        output, expected_keys
    )

    print("\n[3] FINAL CHECKPOINT VALIDATION", flush=True)

    for pct in RETENTIONS:
        count = sum(
            key[1] == pct for key in completed
        )

        print(
            f"{pct}%: {count} predictions",
            flush=True,
        )

    assert set(completed) == expected_keys, (
        "Not all requested predictions completed."
    )

    print("\n" + "=" * 76, flush=True)
    print("CELL 12 INFERENCE COMPLETE", flush=True)
    print("Predictions:", len(completed), flush=True)
    print("Checkpoint:", output, flush=True)
    print("=" * 76, flush=True)


if __name__ == "__main__":
    main()
'''

RUNNER.parent.mkdir(parents=True, exist_ok=True)

RUNNER.write_text(
    textwrap.dedent(runner_code).lstrip(),
    encoding="utf-8",
)

print("\n[1] VERIFY RUNNER SYNTAX")

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(RUNNER)],
    check=True,
)

print("Syntax: PASS")

print("\n[2] EXECUTE SIX-PREDICTION PILOT\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])
env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [
        str(PYTHON), "-u", str(RUNNER),
        "--mode", "pilot",
    ],
    cwd=str(REPO),
    env=env,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "Cell 12 pilot failed. "
        "Share the traceback before proceeding."
    )

print("\n[3] INDEPENDENT PILOT INTEGRITY CHECK")

with PILOT.open("r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(records) == 6

pairs = {
    (str(r["qid"]), int(r["retention_pct"]))
    for r in records
}

assert len(pairs) == 6

for pct in (75, 50, 25):
    subset = [
        r for r in records
        if r["retention_pct"] == pct
    ]
    assert len(subset) == 2
    assert all(
        r["retained_visual_tokens"] == 576 * pct // 100
        for r in subset
    )

# Confirm the first question reproduces
# previously validated answers.
expected_q11934 = {
    75: "The image is taken using a computed tomography (CT) scan.",
    50: "The image is taken using a computed tomography (CT) scan.",
    25: "The image is taken using a chest computed tomography (CT) scan.",
}

first_question = [
    r for r in records
    if str(r["qid"]) == "11934"
]

assert len(first_question) == 3

for record in first_question:
    pct = record["retention_pct"]

    assert record["prediction"] == expected_q11934[pct], (
        f"Pilot prediction differs from Cell 10 "
        f"at retention {pct}%"
    )

print("Six predictions: PASS")
print("All retention counts: PASS")
print("Unique question/retention pairs: PASS")
print("QID 11934 answer reproduction: PASS")
print("B1-compatible scoring fields: PASS")

PILOT_REPORT.write_text(
    json.dumps({
        "experiment": "B2_FastV",
        "status": "PASS",
        "mode": "pilot",
        "questions": 2,
        "retentions": [75, 50, 25],
        "prediction_count": 6,
        "checkpoint": str(PILOT),
        "runner": str(RUNNER),
        "scoring": "Exact B1 Cell 11B",
    }, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 76)
print("B2 — CELL 12 PILOT PASSED")
print("=" * 76)
print("Runner:", RUNNER)
print("Pilot checkpoint:", PILOT)
print("Pilot report:", PILOT_REPORT)


B2 — CELL 12: RESUMABLE INFERENCE RUNNER

[1] VERIFY RUNNER SYNTAX
Syntax: PASS

[2] EXECUTE SIX-PREDICTION PILOT

B2 — FASTV CHECKPOINTED INFERENCE
Mode: pilot
Questions: 2
Retentions: (75, 50, 25)
Expected predictions: 6
Already completed: 0
Remaining: 6
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_pilot_predictions.jsonl

[1] LOAD FROZEN MODEL
B2 — FROZEN TWO-GPU LLAVA-MED LOAD

[1] TOKENIZER
Vocabulary: 32000

[2] LOAD ORIGINAL CHECKPOINT


Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.01s/it]


missing_keys: 0
unexpected_keys: 0
mismatched_keys: 0
error_msgs: 0

[3] VERIFY GPU PLACEMENT
model.embed_tokens: cuda:0 — PASS
model.vision_tower: cuda:0 — PASS
model.mm_projector: cuda:0 — PASS
model.layers.0: cuda:0 — PASS
model.layers.15: cuda:0 — PASS
model.layers.16: cuda:1 — PASS
model.layers.31: cuda:1 — PASS
model.norm: cuda:1 — PASS
lm_head: cuda:1 — PASS

[4] VERIFY VISION WEIGHTS
Vision weight max absolute error: 0.0
Original vision weights: EXACT MATCH

[5] FREEZE MODEL
Parameters: 7566219264
Trainable: 0

[6] INITIALIZE IMAGE PROCESSOR
Vision patches: 576 — PASS

[7] GPU MEMORY
GPU 0: 7.599 GiB allocated
GPU 1: 6.994 GiB allocated

Saved: /kaggle/working/B2_FastV/configs/model_load_report.json

CELL 4 MODEL VERIFICATION PASSED

[2] RUN INFERENCE

Question 1/2 | QID=11934 | pending=[75, 50, 25]
  75% | 16 tokens | prefill=587.27ms | decode=1320.95ms
  Answer: 'The image is taken using a computed tomography (CT) scan.'
  Saved 1/6
  50% | 16 tokens | prefill=255.96ms | deco

In [15]:

# ============================================================
# B2 — CELL 13: CHECKPOINT INTEGRITY AND RESUME TEST
# ============================================================

from pathlib import Path
from collections import Counter
import subprocess
import hashlib
import json
import os
import sys

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

RUNNER = ROOT / "scripts/b2_fastv_inference_runner.py"
PILOT = ROOT / "predictions/b2_fastv_pilot_predictions.jsonl"
MANIFEST = ROOT / "manifests/b2_fastv_3183_manifest.jsonl"
REPORT = ROOT / "configs/b2_checkpoint_resume_test.json"

for path in (PYTHON, RUNNER, PILOT, MANIFEST,
             ROOT / "b2_metrics.py"):
    assert path.is_file(), f"Missing: {path}"

sys.path.insert(0, str(ROOT))
from b2_metrics import evaluate_answer

print("=" * 76)
print("B2 — CELL 13: CHECKPOINT INTEGRITY AND RESUME")
print("=" * 76)

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f if line.strip()
        ]

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ------------------------------------------------------------
# 1. Read pilot and official manifest
# ------------------------------------------------------------

print("\n[1] READ PILOT AND MANIFEST")

records = read_jsonl(PILOT)
manifest = read_jsonl(MANIFEST)

assert len(records) == 6
assert len(manifest) == 3183

manifest_by_key = {
    (str(r["qid"]), int(r["retention_pct"])): r
    for r in manifest
}
assert len(manifest_by_key) == 3183

pilot_keys = [
    (str(r["qid"]), int(r["retention_pct"]))
    for r in records
]

assert len(set(pilot_keys)) == 6

expected_qids = {
    str(manifest[0]["qid"]),
    str(manifest[3]["qid"]),
}

assert {key[0] for key in pilot_keys} == expected_qids
assert {key[1] for key in pilot_keys} == {75, 50, 25}

print("Pilot predictions: 6 — PASS")
print("Official manifest: 3183 — PASS")
print("Unique prediction keys: PASS")

# ------------------------------------------------------------
# 2. Verify record identity and B1 scores
# ------------------------------------------------------------

print("\n[2] VERIFY QUESTION ALIGNMENT AND SCORING")

expected_visual = {75: 432, 50: 288, 25: 144}

for record in records:
    key = (str(record["qid"]), int(record["retention_pct"]))
    assert key in manifest_by_key

    official = manifest_by_key[key]

    for field in (
        "question", "reference", "answer_type",
        "img_name", "original_test_index"
    ):
        assert record[field] == official[field], (
            f"Mismatch in {field} for {key}"
        )

    pct = key[1]
    assert record["retained_visual_tokens"] == expected_visual[pct]
    assert record["pruning_after_layer"] == 2
    assert record["original_visual_tokens"] == 576

    # Recompute with the exact B1 Cell 11B scorer.
    expected_score = evaluate_answer(
        record["answer_type"],
        record["reference"],
        record["prediction"]
    )

    assert record["scoring"] == expected_score, (
        f"Scoring mismatch for {key}"
    )

    original_len = record["original_sequence_length"]
    reduced_len = original_len - 576 + expected_visual[pct]

    assert record["reduced_sequence_length"] == reduced_len
    assert record["prefill_cache_lengths"] == (
        [original_len] * 2 + [reduced_len] * 30
    )

    generated_count = record["generated_token_count"]
    processed_count = len(record["generated_token_ids"]) - 1

    assert 1 <= generated_count <= 64
    assert generated_count == len(record["generated_token_ids"])

    # Last predicted token is not fed through the decoder.
    assert record["final_cache_lengths"] == (
        [original_len + processed_count] * 2
        + [reduced_len + processed_count] * 30
    )

    profile = record["profiling"]

    for field in (
        "llm_prefill_ms",
        "decode_ms",
        "total_inference_ms",
        "gpu0_peak_allocated_gib",
        "gpu1_peak_allocated_gib",
    ):
        assert isinstance(profile[field], (int, float))
        assert profile[field] >= 0

    assert profile["llm_prefill_ms"] > 0
    assert profile["total_inference_ms"] > 0

print("Official question alignment: PASS")
print("B1-compatible scoring recomputation: PASS")
print("Physical token counts and KV caches: PASS")
print("Profiling fields: PASS")

# ------------------------------------------------------------
# 3. Record original checkpoint identity
# ------------------------------------------------------------

print("\n[3] RECORD CHECKPOINT HASH")

before_hash = sha256(PILOT)
before_size = PILOT.stat().st_size

print("SHA-256 before resume:", before_hash)
print("Checkpoint bytes:", before_size)

# ------------------------------------------------------------
# 4. Resume the completed pilot
# ------------------------------------------------------------

print("\n[4] TEST RESUME WITHOUT REGENERATING\n", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", "")
])
env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(RUNNER), "--mode", "pilot"],
    cwd=str(REPO),
    env=env,
    capture_output=True,
    text=True,
    check=False
)

print(result.stdout, flush=True)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Resume verification failed.")

assert "Already completed: 6" in result.stdout
assert "Remaining: 0" in result.stdout

assert "[1] LOAD FROZEN MODEL" not in result.stdout
assert "Loading checkpoint shards" not in result.stdout

print("No redundant model loading: PASS")
print("No redundant inference: PASS")

# ------------------------------------------------------------
# 5. Verify checkpoint has not changed
# ------------------------------------------------------------

print("\n[5] VERIFY CHECKPOINT IMMUTABILITY")

after_hash = sha256(PILOT)
after_size = PILOT.stat().st_size

assert after_hash == before_hash
assert after_size == before_size
assert len(read_jsonl(PILOT)) == 6

print("SHA-256 unchanged: PASS")
print("File size unchanged: PASS")
print("No duplicate predictions: PASS")

# ------------------------------------------------------------
# 6. Save verification report
# ------------------------------------------------------------

print("\n[6] SAVE VERIFICATION REPORT")

REPORT.parent.mkdir(parents=True, exist_ok=True)

report = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "pilot_predictions": 6,
    "official_manifest_predictions": 3183,
    "retentions": [75, 50, 25],
    "question_alignment_verified": True,
    "b1_scoring_verified": True,
    "cache_lengths_verified": True,
    "resume_remaining": 0,
    "redundant_model_loading": False,
    "checkpoint_unchanged": True,
    "pilot_checkpoint_sha256": after_hash,
    "runner": str(RUNNER),
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", REPORT)

print("\n" + "=" * 76)
print("CELL 13 PASSED — CHECKPOINT RESUME VERIFIED")
print("=" * 76)
print("Completed pilot predictions: 6/6")
print("Resume behavior: PASS")
print("B1 metric compatibility: PASS")
print("Checkpoint integrity: PASS")


B2 — CELL 13: CHECKPOINT INTEGRITY AND RESUME

[1] READ PILOT AND MANIFEST
Pilot predictions: 6 — PASS
Official manifest: 3183 — PASS
Unique prediction keys: PASS

[2] VERIFY QUESTION ALIGNMENT AND SCORING
Official question alignment: PASS
B1-compatible scoring recomputation: PASS
Physical token counts and KV caches: PASS
Profiling fields: PASS

[3] RECORD CHECKPOINT HASH
SHA-256 before resume: 3ea8768f468356046ab3cdb3c757f755fdf7daafa0214955091d7e2b0b4e8676
Checkpoint bytes: 10371

[4] TEST RESUME WITHOUT REGENERATING

B2 — FASTV CHECKPOINTED INFERENCE
Mode: pilot
Questions: 2
Retentions: (75, 50, 25)
Expected predictions: 6
Already completed: 6
Remaining: 0
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_pilot_predictions.jsonl

[3] FINAL CHECKPOINT VALIDATION
75%: 2 predictions
50%: 2 predictions
25%: 2 predictions

CELL 12 INFERENCE COMPLETE
Predictions: 6
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_pilot_predictions.jsonl

No redundant model loading: PASS
N

In [17]:

# ============================================================
# B2 — CELL 14A: ROBUST GPU WARM-UP PATCH
# ============================================================

from pathlib import Path
import ast
import hashlib
import json
import os
import subprocess
import sys

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

RUNNER = ROOT / "scripts/b2_fastv_inference_runner.py"
BACKUP = ROOT / "scripts/b2_fastv_inference_runner_pre_cell14.py"
PILOT = ROOT / "predictions/b2_fastv_pilot_predictions.jsonl"
REPORT = ROOT / "configs/b2_profiling_protocol.json"

assert RUNNER.is_file()
assert BACKUP.is_file()
assert PILOT.is_file()
assert PYTHON.is_file()

print("=" * 76)
print("B2 — CELL 14A: ROBUST WARM-UP PATCH")
print("=" * 76)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

pilot_hash_before = sha256(PILOT)

# Always patch the preserved, verified Cell 12 runner.
original = BACKUP.read_text(encoding="utf-8")
tree = ast.parse(original)

functions = {
    node.name: node
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}

assert "main" in functions
assert "infer_one" in functions
assert "prepare_question" in functions

# ------------------------------------------------------------
# 1. Define warm-up function
# ------------------------------------------------------------

print("\n[1] PREPARE WARM-UP FUNCTION")

warmup = '''
def warmup_model(model, tokenizer, tower, sample, image_root):
    """
    Run actual FastV inference before timed full-dataset
    predictions. No warm-up results enter the checkpoint.
    """
    print("\\\\n[WARM-UP] START", flush=True)

    prepared = prepare_question(
        sample, tokenizer, model, tower, image_root
    )

    log = []

    for repeat in range(2):
        for pct in RETENTIONS:
            result = infer_one(
                model, tokenizer, prepared, pct
            )

            log.append({
                "pass": repeat + 1,
                "retention_pct": pct,
                "prefill_ms": result["profiling"]["llm_prefill_ms"],
            })

            print(
                f"[WARM-UP] {repeat+1}/2 | {pct}% | "
                f"prefill={result['profiling']['llm_prefill_ms']:.2f}ms",
                flush=True,
            )

            del result

    del prepared
    gc.collect()
    sync_gpus()

    print("[WARM-UP] COMPLETE: 6 inferences", flush=True)
    return log


'''

ast.parse(warmup)

# ------------------------------------------------------------
# 2. Insert warm-up before the inference loop
# ------------------------------------------------------------

print("\n[2] LOCATE INFERENCE LOOP")

main = functions["main"]
lines = original.splitlines(keepends=True)

# Find the model-loading block by its actual AST
# structure rather than by a literal print statement.
with_blocks = [
    node
    for node in ast.walk(main)
    if isinstance(node, ast.With)
]

# The inference loop is the 'for q_index, question'
# statement inside main(), following model loading.
target_loops = [
    node for node in ast.walk(main)
    if isinstance(node, ast.For)
    and isinstance(node.target, ast.Tuple)
    and any(
        isinstance(element, ast.Name)
        and element.id == "q_index"
        for element in node.target.elts
    )
]

assert len(target_loops) == 1, (
    "Could not uniquely locate question inference loop."
)

loop = target_loops[0]

# The question loop is nested inside:
# with output.open(...) as out:
# Therefore, the warm-up must be placed BEFORE
# the surrounding with statement.
candidate_with = [
    node for node in with_blocks
    if node.lineno < loop.lineno <= node.end_lineno
]

assert len(candidate_with) == 1, (
    "Could not identify the prediction-writing block."
)

write_block = candidate_with[0]

# Main body indentation = 4 spaces, but the writer
# is inside `if pending:` and uses 8 spaces.
indent = lines[write_block.lineno - 1][
    :len(lines[write_block.lineno - 1])
    - len(lines[write_block.lineno - 1].lstrip())
]

assert indent == "        ", (
    f"Unexpected insertion indentation: {indent!r}"
)

insertion = [
    indent + 'if args.mode == "full":\n',
    indent + '    warmup_question = next(\n',
    indent + '        q for q in selected\n',
    indent + '        if any(\n',
    indent + '            (str(q["qid"]), pct) not in completed\n',
    indent + '            for pct in RETENTIONS\n',
    indent + '        )\n',
    indent + '    )\n',
    indent + '    warmup_model(\n',
    indent + '        model, tokenizer, tower,\n',
    indent + '        warmup_question, image_root\n',
    indent + '    )\n',
    '\n',
]

# Insert immediately before the checkpoint writing block.
patched_lines = (
    lines[:write_block.lineno - 1]
    + insertion
    + lines[write_block.lineno - 1:]
)

patched = warmup + "".join(patched_lines)

# ------------------------------------------------------------
# 3. Validate before writing
# ------------------------------------------------------------

print("\n[3] VALIDATE PATCHED RUNNER")

compiled = compile(
    patched,
    str(RUNNER),
    "exec",
)

patched_ast = ast.parse(patched)

assert sum(
    isinstance(node, ast.FunctionDef)
    and node.name == "warmup_model"
    for node in patched_ast.body
) == 1

assert patched.count("warmup_model(") == 2
assert 'if args.mode == "full":' in patched
assert "from b2_metrics import evaluate_answer" in patched
assert "fastv_cached_prefill(" in patched
assert "fastv_cached_decode_step(" in patched
assert "os.fsync(out.fileno())" in patched

RUNNER.write_text(patched, encoding="utf-8")

subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(RUNNER)],
    check=True,
)

print("Syntax: PASS")
print("FastV implementation unchanged: PASS")
print("B1 scoring unchanged: PASS")
print("Warm-up inserted before checkpoint writer: PASS")

# ------------------------------------------------------------
# 4. Run resume-only safety test
# ------------------------------------------------------------

print("\n[4] VERIFY PILOT RESUME", flush=True)

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(ROOT),
    str(REPO),
    env.get("PYTHONPATH", "")
])
env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(
    [str(PYTHON), "-u", str(RUNNER), "--mode", "pilot"],
    cwd=str(REPO),
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)

assert result.returncode == 0, result.stderr
assert "Already completed: 6" in result.stdout
assert "Remaining: 0" in result.stdout
assert "[WARM-UP]" not in result.stdout
assert "[1] LOAD FROZEN MODEL" not in result.stdout

assert sha256(PILOT) == pilot_hash_before

print("Pilot resume: PASS")
print("Checkpoint unchanged: PASS")

# ------------------------------------------------------------
# 5. Save profiling configuration
# ------------------------------------------------------------

print("\n[5] SAVE PROFILING PROTOCOL")

report = {
    "experiment": "B2_FastV",
    "status": "PASS",
    "runner": str(RUNNER),
    "runner_sha256": sha256(RUNNER),
    "warmup_enabled": True,
    "warmup_passes": 2,
    "warmup_retentions": [75, 50, 25],
    "warmup_inferences": 6,
    "warmup_predictions_saved": False,
    "timing": "Synchronized wall-clock GPU inference",
    "image_encoding_timed_separately": True,
    "prefill_timed_separately": True,
    "decoding_timed_separately": True,
    "gpu_memory": "Per-device PyTorch peak allocation",
    "b1_metrics": "Exact B1 Cell 11B",
    "resume_verified": True,
    "pilot_checkpoint_unchanged": True,
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:", REPORT)

print("\n" + "=" * 76)
print("CELL 14A PASSED — WARM-UP AND PROFILING READY")
print("=" * 76)
print("Experimental retentions: 75%, 50%, 25%")
print("Warm-up: 6 unrecorded inferences")
print("Prediction checkpoint: unchanged")
print("Ready for Cell 15")


B2 — CELL 14A: ROBUST WARM-UP PATCH

[1] PREPARE WARM-UP FUNCTION

[2] LOCATE INFERENCE LOOP

[3] VALIDATE PATCHED RUNNER
Syntax: PASS
FastV implementation unchanged: PASS
B1 scoring unchanged: PASS
Warm-up inserted before checkpoint writer: PASS

[4] VERIFY PILOT RESUME
B2 — FASTV CHECKPOINTED INFERENCE
Mode: pilot
Questions: 2
Retentions: (75, 50, 25)
Expected predictions: 6
Already completed: 6
Remaining: 0
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_pilot_predictions.jsonl

[3] FINAL CHECKPOINT VALIDATION
75%: 2 predictions
50%: 2 predictions
25%: 2 predictions

CELL 12 INFERENCE COMPLETE
Predictions: 6
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_pilot_predictions.jsonl

Pilot resume: PASS
Checkpoint unchanged: PASS

[5] SAVE PROFILING PROTOCOL
Saved: /kaggle/working/B2_FastV/configs/b2_profiling_protocol.json

CELL 14A PASSED — WARM-UP AND PROFILING READY
Experimental retentions: 75%, 50%, 25%
Warm-up: 6 unrecorded inferences
Prediction checkpoint: unch

In [18]:

# ============================================================
# B2 — CELL 15: FULL FASTV INFERENCE
#
# Official SLAKE English test: 1,061 questions
# Retentions: 75%, 50%, 25%
# Expected predictions: 3,183
#
# Checkpointed and resumable.
# ============================================================

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import subprocess
import os
import sys
import json
import hashlib

ROOT = Path("/kaggle/working/B2_FastV")
REPO = Path("/kaggle/working/LLaVA-Med")
PYTHON = ROOT / "venv/bin/python"

RUNNER = ROOT / "scripts/b2_fastv_inference_runner.py"

OUTPUT = (
    ROOT / "predictions/b2_fastv_full_predictions.jsonl"
)

REPORT = (
    ROOT / "configs/b2_full_inference_report.json"
)

MANIFEST = (
    ROOT / "manifests/b2_fastv_3183_manifest.jsonl"
)

PROTOCOL = ROOT / "configs/experiment_protocol.json"
PROFILE_CONFIG = ROOT / "configs/b2_profiling_protocol.json"

print("=" * 76)
print("B2 — CELL 15: FULL FASTV INFERENCE")
print("=" * 76)

# ------------------------------------------------------------
# 1. Verify prerequisites
# ------------------------------------------------------------

print("\n[1] VERIFY EXPERIMENT PREREQUISITES")

required = [
    PYTHON,
    RUNNER,
    MANIFEST,
    PROTOCOL,
    PROFILE_CONFIG,
    ROOT / "load_b2_model.py",
    ROOT / "fastv_selection.py",
    ROOT / "fastv_prefill.py",
    ROOT / "fastv_cached.py",
    ROOT / "b2_metrics.py",
    ROOT / "configs/b2_checkpoint_resume_test.json",
    ROOT / "configs/b1_metric_compatibility_test.json",
]

for path in required:
    assert path.is_file(), f"Missing prerequisite: {path}"

config = json.loads(PROTOCOL.read_text(encoding="utf-8"))

assert config["retention_percentages"] == [75, 50, 25]
assert config["expected_predictions"] == 3183
assert config["decoding"]["max_new_tokens"] == 64

profile_config = json.loads(
    PROFILE_CONFIG.read_text(encoding="utf-8")
)

assert profile_config["status"] == "PASS"
assert profile_config["warmup_enabled"] is True

def read_jsonl(path):
    if not path.is_file():
        return []

    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

manifest = read_jsonl(MANIFEST)

assert len(manifest) == 3183

expected_keys = {
    (str(r["qid"]), int(r["retention_pct"]))
    for r in manifest
}

assert len(expected_keys) == 3183

print("Model and FastV modules: FOUND")
print("Exact B1 metric module: FOUND")
print("Warm-up configuration: PASS")
print("Official manifest: 3183 — PASS")

# ------------------------------------------------------------
# 2. Examine existing checkpoint
# ------------------------------------------------------------

print("\n[2] VERIFY RESUMABLE CHECKPOINT")

existing = read_jsonl(OUTPUT)

existing_keys = [
    (str(r["qid"]), int(r["retention_pct"]))
    for r in existing
]

assert len(existing_keys) == len(set(existing_keys)), (
    "Duplicate predictions found in checkpoint."
)

assert set(existing_keys).issubset(expected_keys), (
    "Checkpoint includes unexpected questions or retentions."
)

remaining = 3183 - len(existing)

print("Already completed:", len(existing))
print("Remaining:", remaining)
print("Output:", OUTPUT)

# ------------------------------------------------------------
# 3. Prepare execution environment
# ------------------------------------------------------------

print("\n[3] PREPARE FROZEN INFERENCE ENVIRONMENT")

env = os.environ.copy()

env["PYTHONPATH"] = os.pathsep.join([
    str(REPO),
    str(ROOT),
    env.get("PYTHONPATH", ""),
])

env["HF_HOME"] = "/tmp/b2_llava_med_hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

# Avoid CPU thread oversubscription.
env["OMP_NUM_THREADS"] = "4"
env["MKL_NUM_THREADS"] = "4"

runner_source = RUNNER.read_text(encoding="utf-8")

assert "def warmup_model(" in runner_source
assert "from b2_metrics import evaluate_answer" in runner_source
assert "fastv_cached_prefill(" in runner_source
assert "fastv_cached_decode_step(" in runner_source
assert "os.fsync(out.fileno())" in runner_source

runner_hash = hashlib.sha256(
    RUNNER.read_bytes()
).hexdigest()

print("Runner SHA-256:", runner_hash)
print("Checkpoint/resume enabled: YES")
print("B1-compatible evaluation: YES")
print("FastV retentions: 75%, 50%, 25%")

# ------------------------------------------------------------
# 4. Run full inference
# ------------------------------------------------------------

print("\n[4] LAUNCH FULL INFERENCE", flush=True)

print(
    "Each prediction is saved immediately. "
    "An interrupted run can be resumed by "
    "running this cell again.",
    flush=True
)

print("=" * 76, flush=True)

started = datetime.now(timezone.utc)

try:
    result = subprocess.run(
        [
            str(PYTHON),
            "-u",
            str(RUNNER),
            "--mode",
            "full",
        ],
        cwd=str(REPO),
        env=env,
        check=False,
    )

except KeyboardInterrupt:
    print(
        "\nInference interrupted. Existing completed "
        "predictions remain in the checkpoint."
    )
    print("Checkpoint:", OUTPUT)
    raise

if result.returncode != 0:
    current_count = len(read_jsonl(OUTPUT))

    print("\nINFERENCE STOPPED BEFORE COMPLETION")
    print("Completed predictions:", current_count)
    print("Remaining:", 3183 - current_count)
    print("Checkpoint:", OUTPUT)

    raise RuntimeError(
        "Full inference did not finish. "
        "Inspect the error above before resuming."
    )

# ------------------------------------------------------------
# 5. Independent final integrity verification
# ------------------------------------------------------------

print("\n[5] VERIFY COMPLETE INFERENCE OUTPUT")

records = read_jsonl(OUTPUT)

assert len(records) == 3183, (
    f"Expected 3183 predictions, found {len(records)}"
)

actual_keys = [
    (str(r["qid"]), int(r["retention_pct"]))
    for r in records
]

assert len(set(actual_keys)) == 3183
assert set(actual_keys) == expected_keys

retention_counts = Counter(
    int(r["retention_pct"])
    for r in records
)

assert retention_counts == {
    75: 1061,
    50: 1061,
    25: 1061,
}

expected_visual = {75: 432, 50: 288, 25: 144}

for record in records:
    pct = int(record["retention_pct"])

    assert record["retained_visual_tokens"] == (
        expected_visual[pct]
    )

    assert record["pruning_after_layer"] == 2
    assert record["original_visual_tokens"] == 576

    assert "prediction" in record
    assert "scoring" in record
    assert "profiling" in record

    assert 1 <= record["generated_token_count"] <= 64

    profile = record["profiling"]

    assert profile["llm_prefill_ms"] > 0
    assert profile["decode_ms"] >= 0
    assert profile["total_inference_ms"] > 0

for pct in (75, 50, 25):
    print(
        f"{pct}%: {retention_counts[pct]}/1061 — PASS"
    )

print("Unique records:", len(set(actual_keys)))
print("Missing records: 0")
print("Unexpected retentions: 0")
print("Scoring and profiling fields: PASS")

# ------------------------------------------------------------
# 6. Save completion report
# ------------------------------------------------------------

print("\n[6] SAVE FULL EXPERIMENT REPORT")

finished = datetime.now(timezone.utc)

report = {
    "experiment": "B2_FastV",
    "status": "COMPLETE",
    "dataset": "SLAKE 1.0",
    "split": "official_test",
    "language": "en",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "pruning_after_layer": 2,
    "retentions": [75, 50, 25],
    "retained_visual_tokens": {
        "75": 432,
        "50": 288,
        "25": 144,
    },
    "questions": 1061,
    "predictions": 3183,
    "predictions_per_retention": {
        str(k): v for k, v in retention_counts.items()
    },
    "scoring": "Exact B1 Cell 11B",
    "greedy_max_new_tokens": 64,
    "warmup_enabled": True,
    "checkpoint": str(OUTPUT),
    "runner_sha256": runner_hash,
    "launch_started_utc": started.isoformat(),
    "launch_finished_utc": finished.isoformat(),
    "validation": "PASS",
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", REPORT)

print("\n" + "=" * 76)
print("CELL 15 PASSED — FULL FASTV INFERENCE COMPLETE")
print("=" * 76)
print("Total predictions: 3183 / 3183")
print("75%: 1061")
print("50%: 1061")
print("25%: 1061")
print("B1-compatible scores: SAVED")
print("Prediction checkpoint:", OUTPUT)
print("Completion report:", REPORT)


B2 — CELL 15: FULL FASTV INFERENCE

[1] VERIFY EXPERIMENT PREREQUISITES
Model and FastV modules: FOUND
Exact B1 metric module: FOUND
Warm-up configuration: PASS
Official manifest: 3183 — PASS

[2] VERIFY RESUMABLE CHECKPOINT
Already completed: 0
Remaining: 3183
Output: /kaggle/working/B2_FastV/predictions/b2_fastv_full_predictions.jsonl

[3] PREPARE FROZEN INFERENCE ENVIRONMENT
Runner SHA-256: bfc3928cd84f5fbd51ca09a1287b6542e80620083aa39b8edd636f20c39a54e8
Checkpoint/resume enabled: YES
B1-compatible evaluation: YES
FastV retentions: 75%, 50%, 25%

[4] LAUNCH FULL INFERENCE
Each prediction is saved immediately. An interrupted run can be resumed by running this cell again.
B2 — FASTV CHECKPOINTED INFERENCE
Mode: full
Questions: 1061
Retentions: (75, 50, 25)
Expected predictions: 3183
Already completed: 0
Remaining: 3183
Checkpoint: /kaggle/working/B2_FastV/predictions/b2_fastv_full_predictions.jsonl

[1] LOAD FROZEN MODEL
B2 — FROZEN TWO-GPU LLAVA-MED LOAD

[1] TOKENIZER
Vocabulary: 32

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.97s/it]


missing_keys: 0
unexpected_keys: 0
mismatched_keys: 0
error_msgs: 0

[3] VERIFY GPU PLACEMENT
model.embed_tokens: cuda:0 — PASS
model.vision_tower: cuda:0 — PASS
model.mm_projector: cuda:0 — PASS
model.layers.0: cuda:0 — PASS
model.layers.15: cuda:0 — PASS
model.layers.16: cuda:1 — PASS
model.layers.31: cuda:1 — PASS
model.norm: cuda:1 — PASS
lm_head: cuda:1 — PASS

[4] VERIFY VISION WEIGHTS
Vision weight max absolute error: 0.0
Original vision weights: EXACT MATCH

[5] FREEZE MODEL
Parameters: 7566219264
Trainable: 0

[6] INITIALIZE IMAGE PROCESSOR
Vision patches: 576 — PASS

[7] GPU MEMORY
GPU 0: 7.599 GiB allocated
GPU 1: 6.994 GiB allocated

Saved: /kaggle/working/B2_FastV/configs/model_load_report.json

CELL 4 MODEL VERIFICATION PASSED

[2] RUN INFERENCE
\n[WARM-UP] START
[WARM-UP] 1/2 | 75% | prefill=505.99ms
[WARM-UP] 1/2 | 50% | prefill=254.98ms
[WARM-UP] 1/2 | 25% | prefill=154.14ms
[WARM-UP] 2/2 | 75% | prefill=340.51ms
[WARM-UP] 2/2 | 50% | prefill=257.42ms
[WARM-UP] 2/2 | 2

In [19]:

# ============================================================
# B2 — CELL 16: COMPLETE PREDICTION INTEGRITY AUDIT
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import sys, json, math, hashlib

ROOT = Path("/kaggle/working/B2_FastV")
sys.path.insert(0, str(ROOT))

from b2_metrics import evaluate_answer

PRED = ROOT / "predictions/b2_fastv_full_predictions.jsonl"
MANIFEST = ROOT / "manifests/b2_fastv_3183_manifest.jsonl"
REPORT = ROOT / "configs/b2_cell16_integrity_report.json"

def read_jsonl(path):
    assert path.is_file(), f"Missing: {path}"
    with path.open(encoding="utf-8") as f:
        return [json.loads(s) for s in f if s.strip()]

print("=" * 72)
print("CELL 16 — FULL B2 INTEGRITY AUDIT")
print("=" * 72)

predictions = read_jsonl(PRED)
manifest = read_jsonl(MANIFEST)

assert len(manifest) == 3183
assert len(predictions) == 3183, (
    f"Only {len(predictions)} / 3183 predictions completed."
)

def key(r):
    return str(r["qid"]), int(r["retention_pct"])

locked = {key(r): r for r in manifest}
observed = {key(r): r for r in predictions}

assert len(locked) == 3183
assert len(observed) == 3183, "Duplicate prediction keys."
assert set(locked) == set(observed), "Missing or foreign records."

expected_visual = {75: 432, 50: 288, 25: 144}
counts = Counter()
answer_types = defaultdict(Counter)
max_generation_count = 0
scoring_checks = 0

for k, record in observed.items():
    official = locked[k]
    pct = k[1]

    for field in (
        "question", "reference", "answer_type",
        "img_name", "original_test_index"
    ):
        assert record[field] == official[field], (k, field)

    assert record["retained_visual_tokens"] == expected_visual[pct]
    assert record["original_visual_tokens"] == 576
    assert record["pruning_after_layer"] == 2

    full = int(record["original_sequence_length"])
    short = full - 576 + expected_visual[pct]

    assert record["reduced_sequence_length"] == short
    assert record["prefill_cache_lengths"] == (
        [full] * 2 + [short] * 30
    )

    tokens = record["generated_token_ids"]
    n = int(record["generated_token_count"])

    assert len(tokens) == n
    assert 1 <= n <= 64
    assert isinstance(record["prediction"], str)

    # Prefill provides the first generated-token logits.
    # The last predicted token is not passed back into
    # the decoder, so the cache advances n-1 times.
    processed = n - 1

    assert record["final_cache_lengths"] == (
        [full + processed] * 2
        + [short + processed] * 30
    )

    expected_score = evaluate_answer(
        record["answer_type"],
        record["reference"],
        record["prediction"]
    )

    assert record["scoring"] == expected_score, (
        "B1 scoring discrepancy", k
    )

    p = record["profiling"]
    for field in (
        "llm_prefill_ms", "decode_ms", "total_inference_ms",
        "gpu0_peak_allocated_gib", "gpu1_peak_allocated_gib"
    ):
        assert math.isfinite(float(p[field]))
        assert float(p[field]) >= 0

    assert p["total_inference_ms"] > 0
    assert p["llm_prefill_ms"] > 0
    assert p["image_encoding_excluded"] is True

    counts[pct] += 1
    answer_types[pct][record["answer_type"]] += 1
    max_generation_count = max(max_generation_count, n)
    scoring_checks += 1

for pct in (75, 50, 25):
    assert counts[pct] == 1061
    assert answer_types[pct] == {"closed": 416, "open": 645}

    print(
        f"{pct}%: 1061 valid | "
        "CLOSED=416 | OPEN=645 — PASS"
    )

sha = hashlib.sha256(PRED.read_bytes()).hexdigest()

report = {
    "status": "PASS",
    "prediction_count": 3183,
    "unique_questions": 1061,
    "retentions": [75, 50, 25],
    "counts": dict(counts),
    "answer_types": {
        str(p): dict(c)
        for p, c in answer_types.items()
    },
    "scoring_checks": scoring_checks,
    "max_generated_tokens_observed": max_generation_count,
    "checkpoint_sha256": sha,
    "manifest_alignment": True,
    "heterogeneous_cache_consistency": True,
    "b1_metric_recalculation": True,
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(json.dumps(report, indent=2))

print("\nB1 score recomputation:", scoring_checks, "PASS")
print("KV cache consistency: PASS")
print("Checkpoint SHA-256:", sha)
print("Saved:", REPORT)
print("\nCELL 16 PASSED — ALL 3183 RECORDS VERIFIED")


CELL 16 — FULL B2 INTEGRITY AUDIT
75%: 1061 valid | CLOSED=416 | OPEN=645 — PASS
50%: 1061 valid | CLOSED=416 | OPEN=645 — PASS
25%: 1061 valid | CLOSED=416 | OPEN=645 — PASS

B1 score recomputation: 3183 PASS
KV cache consistency: PASS
Checkpoint SHA-256: 71092631a01a4c4223f30802dabd39e66dd4d92a6567b5c6903fad901a13fe58
Saved: /kaggle/working/B2_FastV/configs/b2_cell16_integrity_report.json

CELL 16 PASSED — ALL 3183 RECORDS VERIFIED


In [20]:

# ============================================================
# B2 — CELL 17: ACCURACY, LATENCY, MEMORY, THROUGHPUT
#
# Outputs:
#   results/b2_performance_summary.csv
#   results/b2_performance_summary.json
#   results/b2_question_level_metrics.csv
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/B2_FastV")
PRED = ROOT / "predictions/b2_fastv_full_predictions.jsonl"
AUDIT = ROOT / "configs/b2_cell16_integrity_report.json"
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

assert json.loads(AUDIT.read_text())["status"] == "PASS"

records = [
    json.loads(line)
    for line in PRED.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(records) == 3183

rows = []

for r in records:
    p = r["profiling"]
    s = r["scoring"]

    # The first token is predicted during prefill.
    # Decode processes n-1 tokens.
    decode_input_tokens = r["generated_token_count"] - 1

    rows.append({
        "qid": str(r["qid"]),
        "retention_pct": int(r["retention_pct"]),
        "answer_type": r["answer_type"],
        "reference": r["reference"],
        "prediction": r["prediction"],
        "closed_accuracy": s["closed_accuracy"],
        "open_precision": s["open_precision"],
        "open_recall": s["open_recall"],
        "open_f1": s["open_f1"],
        "generated_tokens": r["generated_token_count"],
        "decode_input_tokens": decode_input_tokens,
        "eos_terminated": bool(r["eos_terminated"]),
        "image_encoding_ms": r["image_encoding_ms"],
        "prefill_ms": p["llm_prefill_ms"],
        "decode_ms": p["decode_ms"],
        "total_inference_ms": p["total_inference_ms"],
        "gpu0_peak_gib": p["gpu0_peak_allocated_gib"],
        "gpu1_peak_gib": p["gpu1_peak_allocated_gib"],
        "original_sequence_length": r["original_sequence_length"],
        "reduced_sequence_length": r["reduced_sequence_length"],
    })

df = pd.DataFrame(rows)

assert not df.duplicated(["qid", "retention_pct"]).any()
assert sorted(df.retention_pct.unique().tolist()) == [25, 50, 75]

df.to_csv(
    OUT / "b2_question_level_metrics.csv",
    index=False
)

def mean(series):
    return float(series.mean())

def percentile(series, q):
    return float(np.percentile(series, q))

summary = []

for pct in (75, 50, 25):
    x = df[df.retention_pct == pct]
    closed = x[x.answer_type == "closed"]
    opened = x[x.answer_type == "open"]

    assert len(x) == 1061
    assert len(closed) == 416
    assert len(opened) == 645

    total_decode_tokens = int(
        x["decode_input_tokens"].sum()
    )
    total_decode_seconds = (
        x["decode_ms"].sum() / 1000
    )

    summary.append({
        "retention_pct": pct,
        "retained_visual_tokens": 576 * pct // 100,
        "questions": len(x),
        "closed_questions": len(closed),
        "open_questions": len(opened),

        "closed_accuracy_pct":
            100 * mean(closed.closed_accuracy),

        "open_precision_pct":
            100 * mean(opened.open_precision),

        "open_recall_pct":
            100 * mean(opened.open_recall),

        "open_f1_pct":
            100 * mean(opened.open_f1),

        "prefill_mean_ms": mean(x.prefill_ms),
        "prefill_median_ms": float(x.prefill_ms.median()),
        "prefill_p95_ms": percentile(x.prefill_ms, 95),

        "decode_mean_ms": mean(x.decode_ms),
        "decode_median_ms": float(x.decode_ms.median()),

        "total_mean_ms": mean(x.total_inference_ms),
        "total_median_ms":
            float(x.total_inference_ms.median()),
        "total_p95_ms": percentile(x.total_inference_ms, 95),

        "image_encoding_mean_ms":
            mean(x.image_encoding_ms),

        # Includes vision encoding once per question.
        "estimated_image_plus_inference_mean_ms":
            mean(x.image_encoding_ms + x.total_inference_ms),

        "generated_tokens_mean": mean(x.generated_tokens),
        "generated_tokens_total": int(x.generated_tokens.sum()),

        # Aggregate throughput of autoregressive steps;
        # excludes first token predicted by prefill.
        "decode_steps_per_second":
            total_decode_tokens / total_decode_seconds
            if total_decode_seconds > 0 else np.nan,

        "gpu0_peak_allocated_mean_gib":
            mean(x.gpu0_peak_gib),

        "gpu1_peak_allocated_mean_gib":
            mean(x.gpu1_peak_gib),

        "gpu0_peak_allocated_max_gib":
            float(x.gpu0_peak_gib.max()),

        "gpu1_peak_allocated_max_gib":
            float(x.gpu1_peak_gib.max()),

        "eos_termination_pct":
            100 * mean(x.eos_terminated.astype(float)),
    })

summary_df = pd.DataFrame(summary)

summary_csv = OUT / "b2_performance_summary.csv"
summary_json = OUT / "b2_performance_summary.json"

summary_df.to_csv(summary_csv, index=False)
summary_json.write_text(
    json.dumps(
        {
            "status": "PASS",
            "method": "B2 FastV",
            "dataset": "Official SLAKE English test",
            "scoring": "B1 Cell 11B",
            "retentions": summary,
            "notes": {
                "accuracy": "CLOSED exact / leading yes-no",
                "open": "Macro token-overlap P/R/F1",
                "latency": "Synchronized T4 wall-clock",
                "image": "Vision encoding timed separately",
                "memory": "Per-GPU PyTorch allocation, not total energy",
                "power": "Not measured",
            },
        },
        indent=2
    )
)

print("=" * 76)
print("CELL 17 — B2 PERFORMANCE SUMMARY")
print("=" * 76)

columns = [
    "retention_pct",
    "closed_accuracy_pct",
    "open_recall_pct",
    "open_f1_pct",
    "prefill_mean_ms",
    "decode_mean_ms",
    "total_mean_ms",
    "decode_steps_per_second",
]

print(summary_df[columns].round(3).to_string(index=False))

print("\nSaved:")
print(summary_csv)
print(summary_json)
print(OUT / "b2_question_level_metrics.csv")
print("\nCELL 17 PASSED")


CELL 17 — B2 PERFORMANCE SUMMARY
 retention_pct  closed_accuracy_pct  open_recall_pct  open_f1_pct  prefill_mean_ms  decode_mean_ms  total_mean_ms  decode_steps_per_second
            75               47.356           38.855        7.479          425.683        2015.463       2441.152                   11.697
            50               46.875           39.301        7.521          315.235        2021.251       2336.493                   11.705
            25               46.635           39.513        7.746          181.403        2060.521       2241.930                   11.443

Saved:
/kaggle/working/B2_FastV/results/b2_performance_summary.csv
/kaggle/working/B2_FastV/results/b2_performance_summary.json
/kaggle/working/B2_FastV/results/b2_question_level_metrics.csv

CELL 17 PASSED


In [2]:

from pathlib import Path

root = Path("/kaggle/working/B2_FastV")

print("Project directory exists:", root.exists())

if root.exists():
    print("\nRestored project files:\n")
    for item in sorted(root.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)

    checkpoint = root / "predictions/b2_fastv_full_predictions.jsonl"

    print("\nFull inference checkpoint exists:", checkpoint.exists())

    if checkpoint.exists():
        print("Checkpoint size:", round(checkpoint.stat().st_size / 1024 / 1024, 2), "MB")
else:
    print("B2 project directory was not found.")


Project directory exists: True

Restored project files:

📁 __pycache__
📄 b2_metrics.py
📁 checkpoints
📁 configs
📄 fastv_cached.py
📄 fastv_prefill.py
📄 fastv_selection.py
📁 final_results
📄 load_b2_model.py
📁 logs
📁 manifests
📁 metrics
📁 predictions
📁 profiling
📁 results
📁 scripts
📁 venv

Full inference checkpoint exists: True
Checkpoint size: 5.09 MB


In [3]:

from pathlib import Path
import shutil

root = Path("/kaggle/working/B2_FastV")
zip_path = Path("/kaggle/working/B2_FastV_Complete_Results.zip")

# Include results, predictions, scripts, configurations,
# and source code, but exclude the large virtual environment.
with __import__("zipfile").ZipFile(
    zip_path, "w",
    compression=__import__("zipfile").ZIP_DEFLATED
) as archive:
    for file in root.rglob("*"):
        if not file.is_file():
            continue
        if "venv" in file.relative_to(root).parts:
            continue
        if "__pycache__" in file.relative_to(root).parts:
            continue
        archive.write(file, arcname=str(file.relative_to(root)))

print("ZIP created:", zip_path)
print("Size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")


ZIP created: /kaggle/working/B2_FastV_Complete_Results.zip
Size: 0.65 MB
